# 1D Pursuit-Evasion LQ Game — Model-Free Actor-Critic Solver

**Game:** 1D zero-sum linear-quadratic pursuit-evasion game (finite horizon N=5).  
x(k+1) = x(k) + u₁(k) + u₂(k),  u₁ ∈ [−3,3],  u₂ ∈ [−1,1],  x₀ = 2.

**Original paper** (NeurIPS 2024 submission): Solves LQ games using stacked per-step policy networks with known system dynamics (A, B₁, B₂) for gradient backpropagation.

**This work — two key changes:**
1. **Model-free:** Replace dynamics-layer backprop with an Actor-Critic framework (unknown model).
2. **Sinusoidal time embedding:** Replace per-step subnetworks with a single shared network conditioned on time via sinusoidal encoding e(k), reducing parameters and improving generalisation across timesteps.

**Analytical equilibrium (Q=0):**
- Player 1 (minimiser): pure strategy u₁★(k,x) = −sign(x)·min(|x|, c₁)
- Player 2 (maximiser): mixed strategy u₂ ∈ {−1, +1} with equal probability (two-point distribution)
- Game value: V★(x₀=2) = c₂² = 1.0

---
**Notebook structure:**
1. Imports
2. Game environment
3. Network architecture (Actor with time embedding, Double Q-Critic)
4. Replay buffer
5. Config & Trainer skeleton
6. Critic update
7. Actor update & training loop
8. Exploitability (warm-start BR version)
9. Training run
10. Verification & diagnostics

## 1. Imports

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 零和 LQ 博弈 Actor-Critic 框架（完整版）
# 包含：双网络AC训练 + 自适应/固定熵正则 + Exploitability 验证
# ══════════════════════════════════════════════════════════════════════
 
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from dataclasses import dataclass
import matplotlib.pyplot as plt
 
torch.manual_seed(2024)
np.random.seed(2024)
 
# ── GPU 设置（T4 × 2，sm_75，兼容当前 PyTorch）──────────────────────
if torch.cuda.is_available():
    DEVICE    = "cuda"
    N_GPU     = torch.cuda.device_count()
    torch.backends.cudnn.benchmark = True
    print(f"PyTorch {torch.__version__} | Device: {DEVICE} × {N_GPU}")
    for i in range(N_GPU):
        name = torch.cuda.get_device_name(i)
        cap  = torch.cuda.get_device_capability(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU{i}: {name}  sm_{cap[0]}{cap[1]}  {mem:.1f}GB")
else:
    DEVICE = "cpu"
    N_GPU  = 0
    print(f"PyTorch {torch.__version__} | Device: cpu")

## 2. Game Environment

`LQGameEnv` implements the 1D pursuit-evasion dynamics and computes the analytical game value via the piecewise formula from Lemma 2 of the original paper.

The analytical value serves as the ground truth for evaluating convergence (`ValErr = |E[J] − V★|`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 1: 环境
# ══════════════════════════════════════════════════════════════════════
 
class LQGameEnv:
    """
    一维追逃 LQ 博弈环境
    动态：x(k+1) = x(k) + u1(k) + u2(k)
    P1 最小化 J，P2 最大化 J
    对训练器完全黑盒（不暴露 A, B 矩阵）
    """
    def __init__(self, c1=3.0, c2=1.0, N=5,
                 Q=0.0, QN=1.0, R1=0.0, R2=0.0):
        self.c1 = c1
        self.c2 = c2
        self.N  = N
        self.Q  = Q
        self.QN = QN
        self.R1 = R1
        self.R2 = R2
 
    def reset(self, x0: float, batch_size: int, device):
        return torch.full((batch_size, 1), x0,
                          dtype=torch.float32, device=device)
 
    def step(self, x, u1, u2, k):
        x_next = x + u1 + u2
        r = -(self.Q  * x**2
            + self.R1 * u1**2
            - self.R2 * u2**2).squeeze(-1)
        done = (k == self.N - 1)
        if done:
            r = r - self.QN * x_next.squeeze(-1)**2
        return x_next, r, done
 
    def analytic_value(self, x0: float) -> float:
        """支持任意 Q≥0 的解析博弈值"""
        x0_abs    = abs(x0)
        threshold = (self.c1 - self.c2) * (self.N - 1) + self.c1
        if x0_abs <= threshold:
            return self.Q * x0_abs**2 + self.c2**2 * (
                (self.N - 1) * self.Q + self.QN)
        else:
            return (x0_abs + (self.c2 - self.c1) * self.N) ** 2
 
    def analytic_u1(self, x: torch.Tensor) -> torch.Tensor:
        """P1 解析均衡策略（纯策略）"""
        return -x.sign() * x.abs().clamp(max=self.c1)
 
    def analytic_u2_sample(self, x: torch.Tensor) -> torch.Tensor:
        """P2 解析均衡策略（两点分布采样）"""
        signs     = torch.randint(0, 2, x.shape,
                                  device=x.device).float() * 2 - 1
        mixed = -signs * self.c2
        pure  = x.sign() * self.c2
        return torch.where(x.abs() <= self.c1, mixed, pure)
 

## 3. Network Architecture

### Actor (shared across timesteps via time embedding)
Input: [x, z, e(k)] where:
- x ∈ ℝ: current state
- z ~ N(0, I₄): noise (enables mixed strategies)
- e(k) ∈ ℝ⁸: sinusoidal time embedding of normalised time k̄ = k/(N−1)

**Innovation over original paper:** The original paper uses N separate subnetworks (one per timestep). Here, a single network conditioned on e(k) handles all timesteps, reducing parameters by ~N× and allowing the network to share representations across time.

### Double Q-Critic
Input: [x, uᵢ, Γ₋ᵢ, Σ₋ᵢ, e(k)] — crucially takes opponent *moments* (mean + variance), not the full opponent distribution.  
This directly operationalises the SDG dimension-reduction theorem: the optimisation landscape depends on the opponent only through (Γ₋ᵢ, Σ₋ᵢ).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 2: 网络模块
# ══════════════════════════════════════════════════════════════════════
 
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))
 
 
def time_embed(k, N, dim, device):
    """时间步正弦编码"""
    t   = torch.tensor(k / max(N - 1, 1), device=device)
    pos = torch.arange(0, dim, 2, device=device).float()
    enc = torch.zeros(dim, device=device)
    enc[0::2] = torch.sin(t * 10 ** (pos / dim))
    enc[1::2] = torch.cos(t * 10 ** (pos[:dim//2] / dim))
    return enc
 
 
class Actor(nn.Module):
    """
    输入：x(k), z~N(0,I), 时间编码
    输出：u_i ∈ [-action_bound, action_bound]
    噪声输入使 Actor 能表达任意混合策略分布
    """
    def __init__(self, state_dim=1, noise_dim=4,
                 action_bound=1.0, time_dim=8, hidden=64):
        super().__init__()
        self.noise_dim    = noise_dim
        self.action_bound = action_bound
        self.time_dim     = time_dim
        in_dim = state_dim + noise_dim + time_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), Mish(),
            nn.Linear(hidden, hidden), Mish(),
            nn.Linear(hidden, 1),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
 
    def forward(self, x, z, k_emb):
        if k_emb.dim() == 1:
            k_emb = k_emb.unsqueeze(0).expand(x.shape[0], -1)
        return torch.tanh(self.net(
            torch.cat([x, z, k_emb], dim=-1))) * self.action_bound
 
    def sample(self, x, k, N, device):
        B     = x.shape[0]
        z     = torch.randn(B, self.noise_dim, device=device)
        k_emb = time_embed(k, N, self.time_dim, device)
        return self.forward(x, z, k_emb)
 
 
class QCritic(nn.Module):
    """
    输入：x, u_i, Γ_{-i}(对手均值), Σ_{-i}(对手方差), 时间编码
    基于 SDG 结论：优化景观只依赖对手前两阶矩
    """
    def __init__(self, state_dim=1, action_dim=1,
                 opponent_dim=2, time_dim=8, hidden=64):
        super().__init__()
        in_dim = state_dim + action_dim + opponent_dim + time_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), Mish(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), Mish(),
            nn.Linear(hidden, 1),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
 
    def forward(self, x, u, gamma_opp, sigma_opp, k_emb):
        if k_emb.dim() == 1:
            k_emb = k_emb.unsqueeze(0).expand(x.shape[0], -1)
        return self.net(torch.cat(
            [x, u, gamma_opp, sigma_opp, k_emb], dim=-1)).squeeze(-1)
 
 
class DoubleQCritic(nn.Module):
    """双 Critic：mean_Q 用于 TD 目标，conservative_Q 用于 Actor 更新"""
    def __init__(self, **kwargs):
        super().__init__()
        self.c1 = QCritic(**kwargs)
        self.c2 = QCritic(**kwargs)
 
    def forward(self, x, u, gamma_opp, sigma_opp, k_emb):
        return (self.c1(x, u, gamma_opp, sigma_opp, k_emb),
                self.c2(x, u, gamma_opp, sigma_opp, k_emb))
 
    def mean_Q(self, x, u, gamma_opp, sigma_opp, k_emb):
        q1, q2 = self.forward(x, u, gamma_opp, sigma_opp, k_emb)
        return (q1 + q2) / 2.
 
    def conservative_Q(self, x, u, gamma_opp, sigma_opp, k_emb,
                        beta=0.2):
        q1, q2 = self.forward(x, u, gamma_opp, sigma_opp, k_emb)
        return (q1 + q2) / 2. - beta * (q1 - q2).abs()
 
 
def estimate_opponent_moments(actor_opp, x, k, N, device, K=64):
    """Monte Carlo 估计对手条件矩（stop-gradient）"""
    with torch.no_grad():
        B     = x.shape[0]
        x_rep = x.unsqueeze(1).expand(-1, K, -1).reshape(B * K, 1)
        u_rep = actor_opp.sample(x_rep, k, N, device).reshape(B, K, 1)
        return u_rep.mean(dim=1), u_rep.var(dim=1)
 
 

## 4. Replay Buffer

Stores transitions (k, x, u₁, u₂, r, x', done).  
The timestep k is stored explicitly so time embeddings and done flags can be correctly reconstructed during training.  
Mixed sampling: 50% recent (tracks current policy), 50% historical (prevents forgetting).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 3: 经验池
# ══════════════════════════════════════════════════════════════════════
 
class ReplayBuffer:
    """
    存储转移元组 (k, x, u1, u2, r, x_next, done)
    混合采样：recent_ratio 近期 + 其余历史
    """
    def __init__(self, capacity=100_000,
                 recent_ratio=0.5, device=DEVICE):
        self.capacity     = capacity
        self.recent_ratio = recent_ratio
        self.device       = device
        self._k      = np.zeros(capacity,       np.float32)
        self._x      = np.zeros((capacity, 1),  np.float32)
        self._u1     = np.zeros((capacity, 1),  np.float32)
        self._u2     = np.zeros((capacity, 1),  np.float32)
        self._r      = np.zeros(capacity,       np.float32)
        self._x_next = np.zeros((capacity, 1),  np.float32)
        self._done   = np.zeros(capacity,       np.float32)
        self._ptr    = 0
        self._size   = 0
 
    def push(self, k, x, u1, u2, r, x_next, done):
        def to_np(t): return t.detach().cpu().numpy()
        r_np   = to_np(r).flatten()
        n      = len(r_np)
        end    = self._ptr + n
        data   = [
            (self._k,      np.ones(n, np.float32) * float(k)),
            (self._x,      to_np(x)),
            (self._u1,     to_np(u1)),
            (self._u2,     to_np(u2)),
            (self._r,      r_np),
            (self._x_next, to_np(x_next)),
            (self._done,   np.ones(n, np.float32) * float(done)),
        ]
        if end <= self.capacity:
            for arr, d in data:
                arr[self._ptr:end] = d
        else:
            f = self.capacity - self._ptr
            for arr, d in data:
                arr[self._ptr:] = d[:f]
                arr[:n - f]     = d[f:]
        self._ptr  = end % self.capacity
        self._size = min(self._size + n, self.capacity)
 
    def sample(self, batch_size):
        assert self._size >= batch_size, \
            f"Buffer {self._size} < batch {batch_size}"
        n_rec = int(batch_size * self.recent_ratio)
        n_his = batch_size - n_rec
        win   = min(self._size, max(batch_size * 4, 4096))
        if self._ptr >= win:
            ridx = np.arange(self._ptr - win, self._ptr)
        else:
            ridx = np.concatenate([
                np.arange(self.capacity - (win - self._ptr),
                          self.capacity),
                np.arange(0, self._ptr)])
        idx = np.concatenate([
            np.random.choice(ridx, n_rec, replace=True),
            np.random.randint(0, self._size, n_his)])
        np.random.shuffle(idx)
        to_t = lambda a: torch.tensor(a[idx], device=self.device)
        return {k: to_t(v) for k, v in [
            ("k", self._k), ("x", self._x),
            ("u1", self._u1), ("u2", self._u2),
            ("r", self._r), ("x_next", self._x_next),
            ("done", self._done)]}
 
    def __len__(self): return self._size
 

## 5. Config & Trainer Skeleton

Key hyperparameters:
- `noise_dim = 4`, `time_embed_dim = 8`
- Actor lr: 5×10⁻⁵ (slow — avoids overshooting the Nash equilibrium)
- Critic lr: 3×10⁻⁴ (fast — critic must track a non-stationary target)
- `tau = 0.005`: soft target network update
- `beta_cons = 0.2`: conservative Q penalty (prevents actor exploiting critic errors)
- `lambda_entropy = 0.3`: entropy bonus for Player 2 only (encourages mixed strategy)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 4: Config + Trainer 骨架
# ══════════════════════════════════════════════════════════════════════
 
@dataclass
class Config:
    # 环境
    x0:                float = 2.0
    c1:                float = 3.0
    c2:                float = 1.0
    N:                 int   = 5
    Q:                 float = 0.1
    QN:                float = 1.0
    R1:                float = 0.0
    R2:                float = 0.0
    x0_noise:          float = 0.0
    # 网络
    noise_dim:         int   = 4
    time_dim:          int   = 8
    hidden:            int   = 64
    # 经验池
    K_moment:          int   = 128
    buffer_capacity:   int   = 100_000
    recent_ratio:      float = 0.5
    # 训练（GPU 版适当增大 batch，充分利用并行）
    warmup_episodes:   int   = 200
    warmup_batch:      int   = 64
    warmup_critic_steps: int = 500
    total_episodes:    int   = 8000
    train_batch:       int   = 64
    critic_batch:      int   = 1024
    actor_batch:       int   = 1024
    critic_updates:    int   = 20
    actor_updates:     int   = 2
    beta_ucb:          float = 0.2
    actor_lr:          float = 5e-5
    critic_lr:         float = 3e-4
    tau:               float = 0.005
    # 熵正则（固定系数）
    # P1 均衡是纯策略 → 不加熵正则，避免偏离真实均衡
    # P2 均衡是混合策略 → 加熵正则防止坍缩，收敛后梯度自然趋 0
    entropy_coef_p1:   float = 0.0
    entropy_coef_p2:   float = 0.3
    # Exploitability
    br_episodes:       int   = 800
    br_critic_steps:   int   = 400
    br_actor_lr:       float = 1e-4
    br_critic_lr:      float = 3e-4
    br_critic_updates: int   = 20
    br_eval_episodes:  int   = 2000
    br_converge_window:int   = 50
    br_converge_tol:   float = 0.01
    br_eval_freq:      int   = 20
    # 日志
    log_every:         int   = 200
    eval_episodes:     int   = 2000
    device:            str   = DEVICE
    # 双 GPU：P1 用 GPU0，P2 用 GPU1（网络小，跨 GPU 并行意义不大，
    # 此处实现为：两个玩家的网络分别放在不同 GPU，数据在各自 GPU 上）
    use_dual_gpu:      bool  = (N_GPU >= 2)
 
    @property
    def min_buffer_size(self) -> int:
        return self.warmup_episodes * self.warmup_batch * self.N // 2
 
 
class Trainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        dev      = cfg.device
 
        # 双 GPU 时：P1 网络放 GPU0，P2 网络放 GPU1
        # 单 GPU 或 CPU 时两个玩家都在同一设备
        self.dev1 = "cuda:0" if cfg.use_dual_gpu else dev
        self.dev2 = "cuda:1" if cfg.use_dual_gpu else dev
 
        self.env = LQGameEnv(
            c1=cfg.c1, c2=cfg.c2, N=cfg.N,
            Q=cfg.Q, QN=cfg.QN, R1=cfg.R1, R2=cfg.R2)
        self.actor1 = Actor(noise_dim=cfg.noise_dim,
                            action_bound=cfg.c1,
                            time_dim=cfg.time_dim,
                            hidden=cfg.hidden).to(self.dev1)
        self.actor2 = Actor(noise_dim=cfg.noise_dim,
                            action_bound=cfg.c2,
                            time_dim=cfg.time_dim,
                            hidden=cfg.hidden).to(self.dev2)
        self.dc1 = DoubleQCritic(time_dim=cfg.time_dim,
                                  hidden=cfg.hidden).to(self.dev1)
        self.dc2 = DoubleQCritic(time_dim=cfg.time_dim,
                                  hidden=cfg.hidden).to(self.dev2)
        self.dc1_target = copy.deepcopy(self.dc1)
        self.dc2_target = copy.deepcopy(self.dc2)
        for p in self.dc1_target.parameters():
            p.requires_grad_(False)
        for p in self.dc2_target.parameters():
            p.requires_grad_(False)
        self.opt_a1 = Adam(self.actor1.parameters(), lr=cfg.actor_lr)
        self.opt_a2 = Adam(self.actor2.parameters(), lr=cfg.actor_lr)
        self.opt_c1 = Adam(self.dc1.parameters(),    lr=cfg.critic_lr)
        self.opt_c2 = Adam(self.dc2.parameters(),    lr=cfg.critic_lr)
 
        # ReplayBuffer 存在 GPU0（主设备），需要时迁移到对应设备
        self.buffer = ReplayBuffer(
            capacity=cfg.buffer_capacity,
            recent_ratio=cfg.recent_ratio,
            device=self.dev1)
 
        if cfg.use_dual_gpu:
            print(f"  双 GPU 模式：P1→{self.dev1}  P2→{self.dev2}")
 
        self.history = {k: [] for k in [
            "episode", "critic1_loss", "critic2_loss",
            "actor1_loss", "actor2_loss", "value_error",
            "p1_u0_mean", "p1_u0_std",
            "p2_dist_mean", "p2_dist_var",
        ]}
 
    def _soft_update(self):
        tau = self.cfg.tau
        for online, target in [(self.dc1, self.dc1_target),
                                (self.dc2, self.dc2_target)]:
            for p_o, p_t in zip(online.parameters(),
                                 target.parameters()):
                p_t.data.mul_(1.0 - tau)
                p_t.data.add_(tau * p_o.data)
 
    def _collect_trajectory(self, use_random=False):
        cfg = self.cfg
        dev = self.dev1   # 轨迹数据收集在主设备
        B   = cfg.warmup_batch if use_random else cfg.train_batch
        x   = torch.full((B, 1), cfg.x0,
                         dtype=torch.float32, device=dev)
        for k in range(self.env.N):
            if use_random:
                u1 = (torch.rand(B, 1, device=dev)*2-1)*cfg.c1
                u2 = (torch.rand(B, 1, device=dev)*2-1)*cfg.c2
            else:
                u1 = self.actor1.sample(x, k, self.env.N, dev)
                # actor2 在 dev2 上，采样后迁移到 dev1
                u2 = self.actor2.sample(
                    x.to(self.dev2), k, self.env.N,
                    self.dev2).to(dev)
            x_next, r, done = self.env.step(x, u1, u2, k)
            self.buffer.push(k, x, u1, u2, r, x_next, done)
            x = x_next
 
    def warmup(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 0: Warmup")
        print(f"  随机轨迹: {cfg.warmup_episodes} episodes "
              f"× batch{cfg.warmup_batch}")
        for _ in range(cfg.warmup_episodes):
            self._collect_trajectory(use_random=True)
        print(f"  Buffer size: {len(self.buffer):,}")
        log_interval = max(cfg.warmup_critic_steps // 5, 1)
        for step in range(cfg.warmup_critic_steps):
            l1 = self._update_critic(player=1)
            l2 = self._update_critic(player=2)
            self._soft_update()
            if (step + 1) % log_interval == 0:
                print(f"    Warmup [{step+1:4d}/{cfg.warmup_critic_steps}]"
                      f"  C1={l1:.4f}  C2={l2:.4f}")
        print("  Warmup complete.\n")
 
    # 占位，由外部函数覆盖
    def _update_critic(self, player: int) -> float: return 0.0
    def _update_actor(self, player: int) -> float:  return 0.0
    def evaluate(self) -> dict:
        return {"J_mean": 0.0, "value_error": 0.0,
                "p1_u0_mean": 0.0, "p1_u0_std": 0.0,
                "p2_u0_mean": 0.0, "p2_u0_std": 0.0}

## 6. Critic Update

TD target uses `Q_mean` (not `Q_cons`) to avoid systematic negative bias.  
Opponent moments (Γ₋ᵢ, Σ₋ᵢ) are estimated via K=128 Monte Carlo samples under `torch.no_grad()` — stop-gradient prevents leakage to opponent's network.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 5: Critic 更新
# ══════════════════════════════════════════════════════════════════════
 
def _update_critic(self, player: int) -> float:
    cfg = self.cfg
    dev = self.dev1 if player == 1 else self.dev2
    batch   = self.buffer.sample(cfg.critic_batch)
 
    # 数据迁移到对应玩家的设备
    k_float = batch["k"].to(dev)
    x       = batch["x"].to(dev)
    r       = batch["r"].to(dev)
    x_next  = batch["x_next"].to(dev)
    done    = batch["done"].to(dev)
 
    if player == 1:
        u_self     = batch["u1"].to(dev)
        actor_opp  = self.actor2
        actor_self = self.actor1
        dev_opp    = self.dev2
        dc_self    = self.dc1
        dc_target  = self.dc1_target
        opt_self   = self.opt_c1
        r_self     = r
    else:
        u_self     = batch["u2"].to(dev)
        actor_opp  = self.actor1
        actor_self = self.actor2
        dev_opp    = self.dev1
        dc_self    = self.dc2
        dc_target  = self.dc2_target
        opt_self   = self.opt_c2
        r_self     = -r
 
    B           = x.shape[0]
    k_int       = k_float.long()
    k_next      = (k_float + 1).long().clamp(max=self.env.N - 1)
    not_done    = (done == 0.0)
 
    emb_cache  = {kv: time_embed(kv, self.env.N, cfg.time_dim, dev)
                  for kv in range(self.env.N)}
    k_emb_cur  = torch.stack([emb_cache[kv.item()] for kv in k_int])
    k_emb_next = torch.stack([emb_cache[kv.item()] for kv in k_next])
 
    with torch.no_grad():
        g_next = torch.zeros(B, 1, device=dev)
        s_next = torch.zeros(B, 1, device=dev)
        u_next = torch.zeros(B, 1, device=dev)
        if not_done.any():
            for k_val in k_next[not_done].unique():
                mask  = not_done & (k_next == k_val)
                x_sub = x_next[mask]
                # 对手在不同设备上采样，结果迁回当前设备
                g, s  = estimate_opponent_moments(
                    actor_opp, x_sub.to(dev_opp),
                    k=k_val.item(), N=self.env.N,
                    device=dev_opp, K=cfg.K_moment)
                g_next[mask] = g.to(dev)
                s_next[mask] = s.to(dev)
                u_next[mask] = actor_self.sample(
                    x_sub, k_val.item(), self.env.N, dev)
        q_next = dc_target.mean_Q(
            x_next, u_next, g_next, s_next, k_emb_next)
        target = r_self + (1.0 - done) * q_next
 
    g_cur = torch.zeros(B, 1, device=dev)
    s_cur = torch.zeros(B, 1, device=dev)
    with torch.no_grad():
        for k_val in k_int.unique():
            mask  = (k_int == k_val)
            g, s  = estimate_opponent_moments(
                actor_opp, x[mask].to(dev_opp),
                k=k_val.item(), N=self.env.N,
                device=dev_opp, K=cfg.K_moment)
            g_cur[mask] = g.to(dev)
            s_cur[mask] = s.to(dev)
 
    q1_pred, q2_pred = dc_self(
        x, u_self, g_cur, s_cur, k_emb_cur)
    loss = (F.huber_loss(q1_pred, target) +
            F.huber_loss(q2_pred, target))
    opt_self.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(dc_self.parameters(), 5.0)
    opt_self.step()
    return loss.item()
 
Trainer._update_critic = _update_critic
 
 
# ══════════════════════════════════════════════════════════════════════
# Step 6: Actor 更新（固定熵系数，P1/P2 各自独立）
# ══════════════════════════════════════════════════════════════════════
 
def _update_actor(self, player: int) -> float:
    cfg = self.cfg
    dev = self.dev1 if player == 1 else self.dev2
    batch      = self.buffer.sample(cfg.actor_batch)
    k_float    = batch["k"].to(dev)
    x          = batch["x"].to(dev)
    B          = x.shape[0]
    k_int_list = k_float.long()
 
    if player == 1:
        actor_self   = self.actor1
        actor_opp    = self.actor2
        dev_opp      = self.dev2
        dc_self      = self.dc1
        opt_self     = self.opt_a1
        entropy_coef = cfg.entropy_coef_p1
    else:
        actor_self   = self.actor2
        actor_opp    = self.actor1
        dev_opp      = self.dev1
        dc_self      = self.dc2
        opt_self     = self.opt_a2
        entropy_coef = cfg.entropy_coef_p2
 
    emb_cache = {kv: time_embed(kv, self.env.N, cfg.time_dim, dev)
                 for kv in range(self.env.N)}
    k_emb_cur = torch.stack(
        [emb_cache[kv.item()] for kv in k_int_list])
 
    with torch.no_grad():
        gamma_opp = torch.zeros(B, 1, device=dev)
        sigma_opp = torch.zeros(B, 1, device=dev)
        for k_val in k_int_list.unique():
            mask  = (k_int_list == k_val)
            g, s  = estimate_opponent_moments(
                actor_opp, x[mask].to(dev_opp),
                k=k_val.item(), N=self.env.N,
                device=dev_opp, K=cfg.K_moment)
            gamma_opp[mask] = g.to(dev)
            sigma_opp[mask] = s.to(dev)
 
    for p in dc_self.parameters():
        p.requires_grad_(False)
    try:
        u_parts, idx_parts = [], []
        for k_val in k_int_list.unique():
            mask  = (k_int_list == k_val)
            idx   = mask.nonzero(as_tuple=True)[0]
            z_sub = torch.randn(
                mask.sum(), actor_self.noise_dim, device=dev)
            u_parts.append(actor_self(
                x[mask], z_sub, emb_cache[k_val.item()]))
            idx_parts.append(idx)
        all_idx = torch.cat(idx_parts)
        all_u   = torch.cat(u_parts)
        inv_idx = torch.empty_like(all_idx)
        inv_idx[all_idx] = torch.arange(B, device=dev)
        u_self  = all_u[inv_idx]
 
        q      = dc_self.conservative_Q(
            x, u_self, gamma_opp, sigma_opp,
            k_emb_cur, beta=cfg.beta_ucb)
        q_loss = -q.mean()
 
        if entropy_coef > 0:
            n_s   = 8
            x_rep = x.unsqueeze(1).expand(
                -1, n_s, -1).reshape(B * n_s, 1)
            k_rep = k_int_list.unsqueeze(1).expand(
                -1, n_s).reshape(B * n_s)
            u_parts_e, idx_parts_e = [], []
            for k_val in k_rep.unique():
                mask  = (k_rep == k_val)
                idx   = mask.nonzero(as_tuple=True)[0]
                z_sub = torch.randn(
                    mask.sum(), actor_self.noise_dim, device=dev)
                u_parts_e.append(actor_self(
                    x_rep[mask], z_sub, emb_cache[k_val.item()]))
                idx_parts_e.append(idx)
            all_idx_e = torch.cat(idx_parts_e)
            all_u_e   = torch.cat(u_parts_e)
            inv_idx_e = torch.empty_like(all_idx_e)
            inv_idx_e[all_idx_e] = torch.arange(B * n_s, device=dev)
            u_rep_out = all_u_e[inv_idx_e].reshape(B, n_s, 1)
            u_var          = u_rep_out.var(dim=1).mean()
            entropy_approx = 0.5 * torch.log(
                2 * torch.tensor(math.pi * math.e, device=dev)
                * u_var + 1e-8)
            loss = q_loss - entropy_coef * entropy_approx
        else:
            loss = q_loss
 
        opt_self.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(actor_self.parameters(), 2.0)
        opt_self.step()
 
    finally:
        for p in dc_self.parameters():
            p.requires_grad_(True)
 
    return loss.item()
 
Trainer._update_actor = _update_actor
 

## 7. Actor Update & Training Loop

**Actor loss:** L = −mean(Q_cons)  — both players maximise their own sign-consistent Q.

**Player 2 entropy regularisation:** Without it, Actor2 collapses to a pure strategy. The differential entropy approximation Ĥ ≈ ½log(2πe σ̂²) encourages the two-point mixed strategy to be maintained.

**Implementation detail:** Actions are assembled per-timestep group via `torch.cat` + inverse-index reordering (not in-place `tensor[mask] = val`), which preserves the computation graph for backpropagation.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 7: evaluate + train_v3
# ══════════════════════════════════════════════════════════════════════
 
def evaluate(self) -> dict:
    cfg = self.cfg
    dev = self.dev1
    n   = cfg.eval_episodes
    x        = self.env.reset(cfg.x0, n, dev)
    total_r  = torch.zeros(n, device=dev)
    u1_k0 = u2_k0 = None
    with torch.no_grad():
        for k in range(self.env.N):
            u1 = self.actor1.sample(x, k, self.env.N, dev)
            u2 = self.actor2.sample(
                x.to(self.dev2), k, self.env.N,
                self.dev2).to(dev)
            if k == 0:
                u1_k0 = u1.clone()
                u2_k0 = u2.clone()
            x, r, _ = self.env.step(x, u1, u2, k)
            total_r += r
    J_mean = (-total_r).mean().item()
    V_star = self.env.analytic_value(cfg.x0)
    return {
        "J_mean":      J_mean,
        "value_error": abs(J_mean - V_star),
        "p1_u0_mean":  u1_k0.mean().item(),
        "p1_u0_std":   u1_k0.std().item(),
        "p2_u0_mean":  u2_k0.mean().item(),
        "p2_u0_std":   u2_k0.std().item(),
    }
 
Trainer.evaluate = evaluate
 
 
def train_v3(self):
    cfg    = self.cfg
    dev    = cfg.device
    V_star = self.env.analytic_value(cfg.x0)
    print("=" * 60)
    print("Phase 1: AC Adversarial Training v3")
    print(f"  V* = {V_star:.4f}  tau={cfg.tau}  "
          f"H_p1={cfg.entropy_coef_p1}  H_p2={cfg.entropy_coef_p2}")
    print("=" * 60)
 
    best_val_error    = float("inf")
    best_episode      = 0
    best_actor1_state = None
    best_actor2_state = None
 
    for episode in range(1, cfg.total_episodes + 1):
        self._collect_trajectory(use_random=False)
 
        c1_losses, c2_losses = [], []
        for _ in range(cfg.critic_updates):
            c1_losses.append(self._update_critic(player=1))
            c2_losses.append(self._update_critic(player=2))
        self._soft_update()
 
        a1_losses, a2_losses = [], []
        for _ in range(cfg.actor_updates):
            a1_losses.append(self._update_actor(player=1))
            a2_losses.append(self._update_actor(player=2))
 
        metrics      = self.evaluate()
        val_err_true = abs(metrics["J_mean"] - V_star)
        p1_correct   = metrics["p1_u0_mean"] < -0.5
        j_reasonable = metrics["J_mean"] > V_star * 0.8
 
        if val_err_true < best_val_error and p1_correct and j_reasonable:
            best_val_error    = val_err_true
            best_episode      = episode
            best_actor1_state = copy.deepcopy(self.actor1.state_dict())
            best_actor2_state = copy.deepcopy(self.actor2.state_dict())
 
        self.history["episode"].append(episode)
        self.history["critic1_loss"].append(
            sum(c1_losses) / len(c1_losses))
        self.history["critic2_loss"].append(
            sum(c2_losses) / len(c2_losses))
        self.history["actor1_loss"].append(
            sum(a1_losses) / len(a1_losses))
        self.history["actor2_loss"].append(
            sum(a2_losses) / len(a2_losses))
        self.history["value_error"].append(val_err_true)
        self.history["p1_u0_mean"].append(metrics["p1_u0_mean"])
        self.history["p1_u0_std"].append(metrics["p1_u0_std"])
        self.history["p2_dist_mean"].append(metrics["p2_u0_mean"])
        self.history["p2_dist_var"].append(metrics["p2_u0_std"]**2)
 
        if episode % cfg.log_every == 0:
            marker = " *" if episode == best_episode else ""
            print(f"Ep {episode:5d} | ValErr={val_err_true:.4f} | "
                  f"J={metrics['J_mean']:.4f} | "
                  f"P1(mu={metrics['p1_u0_mean']:+.3f},"
                  f"std={metrics['p1_u0_std']:.3f}) | "
                  f"P2(mu={metrics['p2_u0_mean']:+.3f},"
                  f"std={metrics['p2_u0_std']:.3f}){marker}")
 
    if best_actor1_state is not None:
        print(f"\nRestoring best model from episode {best_episode} "
              f"(ValErr={best_val_error:.4f})")
        self.actor1.load_state_dict(best_actor1_state)
        self.actor2.load_state_dict(best_actor2_state)
        torch.save({
            "episode":   best_episode,
            "val_error": best_val_error,
            "V_star":    V_star,
            "actor1":    best_actor1_state,
            "actor2":    best_actor2_state,
            "config":    cfg,
        }, "best_model_Q01.pt")
        print("Best model saved: best_model_Q01.pt")
    else:
        print("\n⚠️  No valid best model found")
    print("\nTraining complete.")
 
Trainer.train = train_v3
 

## 8. Exploitability — Warm-Start Best Response

Standard exploitability uses a randomly-initialised best-response (BR) network, which can underestimate exploitability if the BR fails to train sufficiently.

**Fix:** Initialise the BR network from the current actor's weights (warm start). Since the current policy is already reasonable, the BR trains faster and converges to a tighter upper bound on exploitability.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 详细验证：确认 Exp=0 是真实 NE 而非 BR 训练失败
# ══════════════════════════════════════════════════════════════════════

def detailed_verification(trainer, n_mc=10000):
    cfg = trainer.cfg
    dev = trainer.dev1
    env = trainer.env
    V_star = env.analytic_value(cfg.x0)

    print("=" * 60)
    print("详细 NE 验证")
    print("=" * 60)
    print(f"  V* = {V_star:.4f}  x0={cfg.x0}")

    # ── 当前策略基准 J ─────────────────────────────────────────
    def eval_J(actor1, actor2, n=n_mc, label=""):
        x       = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                u1 = actor1.sample(x, k, env.N, dev)
                u2 = actor2.sample(
                    x.to(trainer.dev2), k, env.N,
                    trainer.dev2).to(dev) if actor2 is trainer.actor2 \
                    else actor2.sample(x, k, env.N, dev)
                x, r, _ = env.step(x, u1, u2, k)
                total_r += r
        j = (-total_r).mean().item()
        std = (-total_r).std().item()
        if label:
            print(f"  {label}: J={j:.4f} ± {std/n**0.5:.4f}")
        return j

    J_cur = eval_J(trainer.actor1, trainer.actor2,
                   label="当前策略 J(α1*, α2*)")

    # ══ 验证1：解析策略基准 ════════════════════════════════════
    print(f"\n[验证1] 解析策略基准")
    print(f"  理论：P1打u1=-x，P2打±c2两点分布，J应≈V*={V_star:.4f}")

    class AnalyticActor1:
        """P1 解析策略：u1 = -x（纯策略）"""
        def sample(self, x, k, N, device):
            return env.analytic_u1(x)

    class AnalyticActor2:
        """P2 解析策略：±c2 两点分布"""
        def sample(self, x, k, N, device):
            return env.analytic_u2_sample(x)

    J_analytic = eval_J(AnalyticActor1(), AnalyticActor2(),
                        label="解析策略 J(u1*=-x, u2*=±c2)")
    print(f"  误差 |J_analytic - V*| = {abs(J_analytic - V_star):.4f}  "
          f"{'✅' if abs(J_analytic - V_star) < 0.05 else '⚠️'}")

    # ══ 验证2：P1 策略检验（固定 α2*，扫描 P1 策略）══════════
    print(f"\n[验证2] P1 策略扫描（固定 α2*，看当前 α1* 是否最优）")
    print(f"  理论：P1最优策略是 u1=-x，任何偏离都应使 J 增大")

    class FixedP1:
        """固定动作的 P1 策略"""
        def __init__(self, u_val):
            self.u_val = u_val
        def sample(self, x, k, N, device):
            return torch.full_like(x, self.u_val)

    class ScaleP1:
        """u1 = scale * (-x) 的 P1 策略"""
        def __init__(self, scale):
            self.scale = scale
        def sample(self, x, k, N, device):
            return (-x * self.scale).clamp(-cfg.c1, cfg.c1)

    print(f"  {'策略':25s}  {'J':>8s}  {'vs α1*':>10s}")
    print(f"  {'-'*50}")

    # 当前 α1* 的 J
    j_p1_cur = eval_J(trainer.actor1, trainer.actor2)
    print(f"  {'当前 α1*':25s}  {j_p1_cur:>8.4f}  {'基准':>10s}")

    # 解析最优
    j_analytic_p1 = eval_J(AnalyticActor1(), trainer.actor2)
    better = "✅ α1* 更好" if j_p1_cur <= j_analytic_p1 else "⚠️ 解析更好"
    print(f"  {'u1=-x（解析最优）':25s}  {j_analytic_p1:>8.4f}  "
          f"{'Δ='+f'{j_analytic_p1-j_p1_cur:+.4f}':>10s} {better}")

    # 缩放策略
    for scale in [0.5, 0.8, 1.0, 1.2, 1.5]:
        j_s = eval_J(ScaleP1(scale), trainer.actor2)
        flag = "← 比α1*好！" if j_s < j_p1_cur - 0.01 else ""
        print(f"  {f'u1={scale:.1f}×(-x)':25s}  {j_s:>8.4f}  "
              f"{'Δ='+f'{j_s-j_p1_cur:+.4f}':>10s} {flag}")

    # 固定常数策略
    for u_val in [-3.0, -2.0, -1.0, 0.0, 1.0]:
        j_s = eval_J(FixedP1(u_val), trainer.actor2)
        flag = "← 比α1*好！" if j_s < j_p1_cur - 0.01 else ""
        print(f"  {f'u1≡{u_val:.1f}（固定）':25s}  {j_s:>8.4f}  "
              f"{'Δ='+f'{j_s-j_p1_cur:+.4f}':>10s} {flag}")

    # 把验证3这一段替换成下面的版本

    # ══ 验证3：P2 策略检验 ════════════════════════════════════
    print(f"\n[验证3] P2 策略扫描（固定 α1*，看当前 α2* 是否最优）")
    print(f"  理论：P2最优策略是±c2两点分布，任何偏离都应使 J 减小")

    class FixedP2:
        def __init__(self, u_val):
            self.u_val = u_val
        def sample(self, x, k, N, device):
            return torch.full_like(x, self.u_val)

    class BiasedP2:
        def __init__(self, p_pos):
            self.p_pos = p_pos
        def sample(self, x, k, N, device):
            mask = torch.rand_like(x) < self.p_pos
            return torch.where(mask,
                               torch.full_like(x,  cfg.c2),
                               torch.full_like(x, -cfg.c2))

    # ── 先算基准，参数传入避免闭包问题 ────────────────────────
    def eval_J_fix_p1(actor2_cand, baseline, n=n_mc, label=""):
        x       = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                u1 = trainer.actor1.sample(x, k, env.N, dev)
                u2 = actor2_cand.sample(x, k, env.N, dev)
                x, r, _ = env.step(x, u1, u2, k)
                total_r += r
        j = (-total_r).mean().item()
        if label:
            flag = "← 比α2*好！" if j > baseline + 0.01 else ""
            print(f"  {label:30s}  {j:>8.4f}  "
                  f"{'Δ='+f'{j-baseline:+.4f}':>10s} {flag}")
        return j

    class CurrentA2:
        def sample(self, x, k, N, device):
            return trainer.actor2.sample(
                x.to(trainer.dev2), k, N,
                trainer.dev2).to(dev)

    # 先算基准
    j_cur_p2 = eval_J_fix_p1(CurrentA2(), baseline=0,
                               label=None)
    print(f"  {'当前 α2*':30s}  {j_cur_p2:>8.4f}  {'基准':>10s}")
    print(f"  {'-'*55}")

    # 再扫描各策略，传入基准值
    eval_J_fix_p1(AnalyticActor2(), j_cur_p2,
                  label="u2=±c2（解析最优）")
    for u_val in [-1.0, -0.5, 0.0, 0.5, 1.0]:
        eval_J_fix_p1(FixedP2(u_val), j_cur_p2,
                      label=f"u2≡{u_val:.1f}（固定）")
    for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
        eval_J_fix_p1(BiasedP2(p), j_cur_p2,
                      label=f"u2=±c2, P(+c2)={p:.1f}")

    # ══ 验证4：Warm-start BR（解决 BR 训练失败问题）════════════
    print(f"\n[验证4] Warm-start BR（用 α2* 初始化，继续寻找更优策略）")
    print(f"  目的：排除 BR 从随机初始化导致局部最优的可能")

    # 用 α2* 的权重初始化 BR actor
    br_actor2_ws = Actor(
        noise_dim=cfg.noise_dim,
        action_bound=cfg.c2,
        time_dim=cfg.time_dim,
        hidden=cfg.hidden).to(dev)
    # 把 actor2 的权重复制过来（注意设备迁移）
    state = {k: v.to(dev) for k, v in
             trainer.actor2.state_dict().items()}
    br_actor2_ws.load_state_dict(state)

    br_critic_ws = DoubleQCritic(
        time_dim=cfg.time_dim, hidden=cfg.hidden).to(dev)
    br_critic_ws_target = copy.deepcopy(br_critic_ws)
    for p in br_critic_ws_target.parameters():
        p.requires_grad_(False)

    opt_ws_a = Adam(br_actor2_ws.parameters(), lr=cfg.br_actor_lr)
    opt_ws_c = Adam(br_critic_ws.parameters(), lr=cfg.br_critic_lr)

    fixed_actor = trainer.actor1
    fixed_dev   = trainer.dev1
    br_buf_ws   = ReplayBuffer(
        capacity=cfg.buffer_capacity // 5,
        recent_ratio=cfg.recent_ratio, device=dev)

    # 收集数据
    for _ in range(50):
        x = env.reset(cfg.x0, cfg.train_batch, dev)
        for k in range(env.N):
            u1  = fixed_actor.sample(x, k, env.N, dev)
            u2  = br_actor2_ws.sample(x, k, env.N, dev)
            x_n, r, done = env.step(x, u1, u2, k)
            br_buf_ws.push(k, x, u1, u2, r, x_n, done)
            x = x_n

    def soft_update_ws():
        for p_o, p_t in zip(br_critic_ws.parameters(),
                             br_critic_ws_target.parameters()):
            p_t.data.mul_(1.0 - cfg.tau)
            p_t.data.add_(cfg.tau * p_o.data)

    # Warmup critic
    for _ in range(200):
        if len(br_buf_ws) < cfg.critic_batch:
            break
        batch    = br_buf_ws.sample(
            min(cfg.critic_batch, len(br_buf_ws)))
        k_float  = batch["k"]
        x_b      = batch["x"]
        r_b      = batch["r"]
        x_next_b = batch["x_next"]
        done_b   = batch["done"]
        u2_b     = batch["u2"]
        r_self   = -r_b
        B        = x_b.shape[0]
        k_int    = k_float.long()
        k_next   = (k_float+1).long().clamp(max=env.N-1)
        not_done = (done_b == 0.0)
        emb_cache = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                     for kv in range(env.N)}
        k_emb_c = torch.stack([emb_cache[kv.item()] for kv in k_int])
        k_emb_n = torch.stack([emb_cache[kv.item()] for kv in k_next])
        with torch.no_grad():
            g1n = torch.zeros(B, 1, device=dev)
            s1n = torch.zeros(B, 1, device=dev)
            u2n = torch.zeros(B, 1, device=dev)
            if not_done.any():
                for kv in k_next[not_done].unique():
                    mask = not_done & (k_next == kv)
                    g, s = estimate_opponent_moments(
                        fixed_actor, x_next_b[mask].to(fixed_dev),
                        kv.item(), env.N, fixed_dev, cfg.K_moment)
                    g1n[mask] = g.to(dev)
                    s1n[mask] = s.to(dev)
                    u2n[mask] = br_actor2_ws.sample(
                        x_next_b[mask], kv.item(), env.N, dev)
            q_n = br_critic_ws_target.mean_Q(
                x_next_b, u2n, g1n, s1n, k_emb_n)
            tgt = r_self + (1.0 - done_b) * q_n
        g1c = torch.zeros(B, 1, device=dev)
        s1c = torch.zeros(B, 1, device=dev)
        with torch.no_grad():
            for kv in k_int.unique():
                mask = (k_int == kv)
                g, s = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    kv.item(), env.N, fixed_dev, cfg.K_moment)
                g1c[mask] = g.to(dev)
                s1c[mask] = s.to(dev)
        q1, q2 = br_critic_ws(x_b, u2_b, g1c, s1c, k_emb_c)
        loss = F.huber_loss(q1, tgt) + F.huber_loss(q2, tgt)
        opt_ws_c.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(br_critic_ws.parameters(), 5.0)
        opt_ws_c.step()
        soft_update_ws()

    # 继续训练 actor
    J_ws_history = []
    for ep in range(400):
        x = env.reset(cfg.x0, cfg.train_batch, dev)
        for k in range(env.N):
            u1  = fixed_actor.sample(x, k, env.N, dev)
            u2  = br_actor2_ws.sample(x, k, env.N, dev)
            x_n, r, done = env.step(x, u1, u2, k)
            br_buf_ws.push(k, x, u1, u2, r, x_n, done)
            x = x_n
        if len(br_buf_ws) >= cfg.critic_batch:
            for _ in range(10):
                # critic update（简化版）
                batch  = br_buf_ws.sample(cfg.critic_batch)
                k_int2 = batch["k"].long()
                x_b2   = batch["x"]
                u2_b2  = batch["u2"]
                r_self2= -batch["r"]
                B2     = x_b2.shape[0]
                emb_c2 = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                          for kv in range(env.N)}
                ke2    = torch.stack(
                    [emb_c2[kv.item()] for kv in k_int2])
                g1c2   = torch.zeros(B2, 1, device=dev)
                s1c2   = torch.zeros(B2, 1, device=dev)
                with torch.no_grad():
                    for kv in k_int2.unique():
                        mask = (k_int2 == kv)
                        g, s = estimate_opponent_moments(
                            fixed_actor, x_b2[mask].to(fixed_dev),
                            kv.item(), env.N, fixed_dev, cfg.K_moment)
                        g1c2[mask] = g.to(dev)
                        s1c2[mask] = s.to(dev)
                    q_t = br_critic_ws_target.mean_Q(
                        x_b2, u2_b2, g1c2, s1c2, ke2)
                    tgt2 = r_self2 + q_t
                q1_2, q2_2 = br_critic_ws(
                    x_b2, u2_b2, g1c2, s1c2, ke2)
                l2 = (F.huber_loss(q1_2, tgt2) +
                      F.huber_loss(q2_2, tgt2))
                opt_ws_c.zero_grad(); l2.backward()
                nn.utils.clip_grad_norm_(
                    br_critic_ws.parameters(), 5.0)
                opt_ws_c.step()
            soft_update_ws()
            # actor update
            for p in br_critic_ws.parameters():
                p.requires_grad_(False)
            try:
                batch_a = br_buf_ws.sample(cfg.actor_batch)
                ki_a    = batch_a["k"].long()
                xa      = batch_a["x"]
                Ba      = xa.shape[0]
                emb_a   = {kv: time_embed(
                    kv, env.N, cfg.time_dim, dev)
                           for kv in range(env.N)}
                ke_a    = torch.stack(
                    [emb_a[kv.item()] for kv in ki_a])
                go = torch.zeros(Ba, 1, device=dev)
                so = torch.zeros(Ba, 1, device=dev)
                with torch.no_grad():
                    for kv in ki_a.unique():
                        mask = (ki_a == kv)
                        g, s = estimate_opponent_moments(
                            fixed_actor, xa[mask].to(fixed_dev),
                            kv.item(), env.N, fixed_dev,
                            cfg.K_moment)
                        go[mask] = g.to(dev)
                        so[mask] = s.to(dev)
                u_p, i_p = [], []
                for kv in ki_a.unique():
                    mask = (ki_a == kv)
                    idx  = mask.nonzero(as_tuple=True)[0]
                    z    = torch.randn(
                        mask.sum(), br_actor2_ws.noise_dim,
                        device=dev)
                    u_p.append(br_actor2_ws(
                        xa[mask], z, emb_a[kv.item()]))
                    i_p.append(idx)
                ai  = torch.cat(i_p)
                au  = torch.cat(u_p)
                inv = torch.empty_like(ai)
                inv[ai] = torch.arange(Ba, device=dev)
                us  = au[inv]
                q_a = br_critic_ws.conservative_Q(
                    xa, us, go, so, ke_a, beta=cfg.beta_ucb)
                la  = -q_a.mean()
                opt_ws_a.zero_grad(); la.backward()
                nn.utils.clip_grad_norm_(
                    br_actor2_ws.parameters(), 2.0)
                opt_ws_a.step()
            finally:
                for p in br_critic_ws.parameters():
                    p.requires_grad_(True)

        if (ep + 1) % 100 == 0:
            x_e    = env.reset(cfg.x0, 1000, dev)
            tot_r  = torch.zeros(1000, device=dev)
            with torch.no_grad():
                for k in range(env.N):
                    u1_e = fixed_actor.sample(x_e, k, env.N, dev)
                    u2_e = br_actor2_ws.sample(x_e, k, env.N, dev)
                    x_e, r_e, _ = env.step(x_e, u1_e, u2_e, k)
                    tot_r += r_e
            j_ws = (-tot_r).mean().item()
            J_ws_history.append(j_ws)
            print(f"    Warm-start BR ep={ep+1}  J={j_ws:.4f}  "
                  f"{'> α2* ✅' if j_ws > J_cur + 0.01 else '≤ α2* (α2*已最优)'}")

    j_ws_final = J_ws_history[-1] if J_ws_history else float('nan')
    ws_delta2  = max(j_ws_final - J_cur, 0.0)
    print(f"\n  Warm-start BR 最终 J = {j_ws_final:.4f}")
    print(f"  Warm-start δ2        = {ws_delta2:.4f}  "
          f"{'✅ α2* 已是真实最优' if ws_delta2 < 0.05 else '⚠️ α2* 还有提升空间'}")

    # ══ 验证5：各时间步状态和动作分析 ═════════════════════════
    print(f"\n[验证5] 各时间步均值分析")
    print(f"  {'k':>3} | {'E[x(k)]':>10} | "
          f"{'E[u1(k)]':>10} | {'E[u2(k)]':>10} | "
          f"{'E[x(k+1)]':>10}")
    print(f"  {'-'*55}")
    x_t    = env.reset(cfg.x0, n_mc, dev)
    with torch.no_grad():
        for k in range(env.N):
            ex  = x_t.mean().item()
            u1k = trainer.actor1.sample(x_t, k, env.N, dev)
            u2k = trainer.actor2.sample(
                x_t.to(trainer.dev2), k, env.N,
                trainer.dev2).to(dev)
            eu1 = u1k.mean().item()
            eu2 = u2k.mean().item()
            x_t, _, _ = env.step(x_t, u1k, u2k, k)
            ex_next = x_t.mean().item()
            print(f"  {k:>3} | {ex:>10.4f} | "
                  f"{eu1:>10.4f} | {eu2:>10.4f} | "
                  f"{ex_next:>10.4f}")
    print(f"  理论：E[x(1)]=E[u2(0)]≈0，之后保持≈0")

    # ══ 综合判断 ═══════════════════════════════════════════════
    print(f"\n{'='*60}")
    print(f"综合判断")
    print(f"{'='*60}")
    checks = {
        "ValErr < 0.05":
            abs(J_cur - V_star) < 0.05,
        "P1 std < 0.05（纯策略）":
            trainer.evaluate()['p1_u0_std'] < 0.05,
        "P2 std > 0.95（混合策略）":
            trainer.evaluate()['p2_u0_std'] > 0.95,
        "解析 P1 不优于当前 α1*":
            j_analytic_p1 >= j_p1_cur - 0.01,
        "Warm-start δ2 < 0.05":
            ws_delta2 < 0.05,
    }
    all_pass = True
    for name, passed in checks.items():
        print(f"  {'✅' if passed else '❌'} {name}")
        if not passed:
            all_pass = False
    print(f"\n  {'🎉 所有验证通过，Exp=0 是真实 NE' if all_pass else '⚠️ 存在未通过项，需要进一步分析'}")
    return checks


# ── 执行 ──────────────────────────────────────────────────────────
checks = detailed_verification(trainer)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Exploitability（Warm-start BR 版本）
# 用当前策略初始化 BR，比随机初始化更可靠
# 完全不依赖解析解
# ══════════════════════════════════════════════════════════════════════

def compute_best_response_warmstart(fixed_actor, br_actor_init,
                                     fixed_player, env, cfg,
                                     verbose=False):
    """
    Warm-start BR：用 br_actor_init 的权重初始化 BR actor。
    
    相比随机初始化的优势：
      - 起点已经是合理策略，不会陷入远离均衡的局部最优
      - 更快收敛，更紧的 Exp 下界
      - 完全不依赖解析解
    
    返回：(br_actor, br_J, J_history, converged)
    """
    dev       = cfg.device
    br_player = 3 - fixed_player
    fixed_dev = next(fixed_actor.parameters()).device

    # ── Warm-start：复制当前策略权重 ─────────────────────────
    br_actor = Actor(
        noise_dim=cfg.noise_dim,
        action_bound=cfg.c1 if br_player == 1 else cfg.c2,
        time_dim=cfg.time_dim,
        hidden=cfg.hidden).to(dev)
    state = {k: v.to(dev)
             for k, v in br_actor_init.state_dict().items()}
    br_actor.load_state_dict(state)

    br_critic = DoubleQCritic(
        time_dim=cfg.time_dim, hidden=cfg.hidden).to(dev)
    br_critic_target = copy.deepcopy(br_critic)
    for p in br_critic_target.parameters():
        p.requires_grad_(False)

    opt_br_a = Adam(br_actor.parameters(),  lr=cfg.br_actor_lr)
    opt_br_c = Adam(br_critic.parameters(), lr=cfg.br_critic_lr)
    br_buf   = ReplayBuffer(
        capacity=cfg.buffer_capacity // 5,
        recent_ratio=cfg.recent_ratio, device=dev)

    def soft_update_br():
        for p_o, p_t in zip(br_critic.parameters(),
                             br_critic_target.parameters()):
            p_t.data.mul_(1.0 - cfg.tau)
            p_t.data.add_(cfg.tau * p_o.data)

    def collect(use_random=False):
        B = cfg.warmup_batch if use_random else cfg.train_batch
        x = env.reset(cfg.x0, B, dev)
        for k in range(env.N):
            if br_player == 1:
                u_br  = ((torch.rand(B, 1, device=dev)*2-1)*cfg.c1
                          if use_random else
                          br_actor.sample(x, k, env.N, dev))
                u_fix = fixed_actor.sample(
                    x.to(fixed_dev), k, env.N, fixed_dev).to(dev)
                u1, u2 = u_br, u_fix
            else:
                u_fix = fixed_actor.sample(
                    x.to(fixed_dev), k, env.N, fixed_dev).to(dev)
                u_br  = ((torch.rand(B, 1, device=dev)*2-1)*cfg.c2
                          if use_random else
                          br_actor.sample(x, k, env.N, dev))
                u1, u2 = u_fix, u_br
            x_next, r, done = env.step(x, u1, u2, k)
            br_buf.push(k, x, u1, u2, r, x_next, done)
            x = x_next

    def update_br_critic():
        if len(br_buf) < cfg.critic_batch:
            return 0.0
        batch    = br_buf.sample(cfg.critic_batch)
        k_float  = batch["k"]
        x_b      = batch["x"]
        r_b      = batch["r"]
        x_next_b = batch["x_next"]
        done_b   = batch["done"]
        B        = x_b.shape[0]
        u_self_b = batch["u1"] if br_player == 1 else batch["u2"]
        r_self   = r_b if br_player == 1 else -r_b
        k_int    = k_float.long()
        k_next   = (k_float + 1).long().clamp(max=env.N - 1)
        not_done = (done_b == 0.0)
        emb_cache  = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                      for kv in range(env.N)}
        k_emb_cur  = torch.stack(
            [emb_cache[kv.item()] for kv in k_int])
        k_emb_next = torch.stack(
            [emb_cache[kv.item()] for kv in k_next])
        with torch.no_grad():
            g_next = torch.zeros(B, 1, device=dev)
            s_next = torch.zeros(B, 1, device=dev)
            u_next = torch.zeros(B, 1, device=dev)
            if not_done.any():
                for k_val in k_next[not_done].unique():
                    mask  = not_done & (k_next == k_val)
                    x_sub = x_next_b[mask]
                    g, s  = estimate_opponent_moments(
                        fixed_actor, x_sub.to(fixed_dev),
                        k=k_val.item(), N=env.N,
                        device=fixed_dev, K=cfg.K_moment)
                    g_next[mask] = g.to(dev)
                    s_next[mask] = s.to(dev)
                    u_next[mask] = br_actor.sample(
                        x_sub, k_val.item(), env.N, dev)
            q_next = br_critic_target.mean_Q(
                x_next_b, u_next, g_next, s_next, k_emb_next)
            target = r_self + (1.0 - done_b) * q_next
        g_cur = torch.zeros(B, 1, device=dev)
        s_cur = torch.zeros(B, 1, device=dev)
        with torch.no_grad():
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                g, s  = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    k=k_val.item(), N=env.N,
                    device=fixed_dev, K=cfg.K_moment)
                g_cur[mask] = g.to(dev)
                s_cur[mask] = s.to(dev)
        q1_p, q2_p = br_critic(
            x_b, u_self_b, g_cur, s_cur, k_emb_cur)
        loss = (F.huber_loss(q1_p, target) +
                F.huber_loss(q2_p, target))
        opt_br_c.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(br_critic.parameters(), 5.0)
        opt_br_c.step()
        return loss.item()

    def update_br_actor():
        if len(br_buf) < cfg.actor_batch:
            return 0.0
        batch  = br_buf.sample(cfg.actor_batch)
        k_int  = batch["k"].long()
        x_b    = batch["x"]
        B      = x_b.shape[0]
        emb_cache = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                     for kv in range(env.N)}
        k_emb_cur = torch.stack(
            [emb_cache[kv.item()] for kv in k_int])
        with torch.no_grad():
            g_opp = torch.zeros(B, 1, device=dev)
            s_opp = torch.zeros(B, 1, device=dev)
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                g, s  = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    k=k_val.item(), N=env.N,
                    device=fixed_dev, K=cfg.K_moment)
                g_opp[mask] = g.to(dev)
                s_opp[mask] = s.to(dev)
        for p in br_critic.parameters():
            p.requires_grad_(False)
        try:
            u_parts, idx_parts = [], []
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                idx   = mask.nonzero(as_tuple=True)[0]
                z_sub = torch.randn(
                    mask.sum(), br_actor.noise_dim, device=dev)
                u_parts.append(br_actor(
                    x_b[mask], z_sub, emb_cache[k_val.item()]))
                idx_parts.append(idx)
            all_idx = torch.cat(idx_parts)
            all_u   = torch.cat(u_parts)
            inv_idx = torch.empty_like(all_idx)
            inv_idx[all_idx] = torch.arange(B, device=dev)
            u_self  = all_u[inv_idx]
            q    = br_critic.conservative_Q(
                x_b, u_self, g_opp, s_opp,
                k_emb_cur, beta=cfg.beta_ucb)
            loss = -q.mean()
            opt_br_a.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(br_actor.parameters(), 2.0)
            opt_br_a.step()
        finally:
            for p in br_critic.parameters():
                p.requires_grad_(True)
        return loss.item()

    def eval_br_J(n=500):
        x_eval  = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                if br_player == 1:
                    u1 = br_actor.sample(x_eval, k, env.N, dev)
                    u2 = fixed_actor.sample(
                        x_eval.to(fixed_dev), k, env.N,
                        fixed_dev).to(dev)
                else:
                    u1 = fixed_actor.sample(
                        x_eval.to(fixed_dev), k, env.N,
                        fixed_dev).to(dev)
                    u2 = br_actor.sample(x_eval, k, env.N, dev)
                x_eval, r, _ = env.step(x_eval, u1, u2, k)
                total_r += r
        return (-total_r).mean().item()

    # ── Warmup（用当前策略收集初始数据）─────────────────────
    for _ in range(min(cfg.warmup_episodes // 2, 50)):
        collect(use_random=False)   # ← Warm-start 用策略数据而非随机
    for _ in range(cfg.br_critic_steps):
        update_br_critic()
        soft_update_br()

    # ── 主循环（含收敛检测）─────────────────────────────────
    J_history = []
    converged = False
    w         = max(cfg.br_converge_window // 20, 1)

    for ep in range(cfg.br_episodes):
        collect(use_random=False)
        for _ in range(cfg.br_critic_updates):
            update_br_critic()
        soft_update_br()
        update_br_actor()

        if (ep + 1) % 20 == 0:
            j_val = eval_br_J()
            J_history.append(j_val)
            if len(J_history) >= w:
                recent  = J_history[-w:]
                j_range = max(recent) - min(recent)
                if j_range < cfg.br_converge_tol:
                    converged = True
                    if verbose:
                        print(f"    BR 收敛于 ep {ep+1}  "
                              f"J={j_val:.4f}  "
                              f"range={j_range:.4f}")
                    break

        if verbose and (ep + 1) % 200 == 0 and not converged:
            j_val = J_history[-1] if J_history else float("nan")
            print(f"    BR ep {ep+1}/{cfg.br_episodes}  "
                  f"J={j_val:.4f}")

    if verbose and not converged:
        print(f"    ⚠️  BR 未收敛（{cfg.br_episodes} ep）"
              f"  建议增大 br_episodes")

    br_J = eval_br_J(n=cfg.br_eval_episodes)
    return br_actor, br_J, J_history, converged


def compute_exploitability_warmstart(trainer, verbose=True) -> dict:
    """
    Warm-start Exploitability：完全不依赖解析解的通用 NE 验证指标。

    与随机初始化版本的区别：
      - BR actor 用当前策略初始化，而非随机初始化
      - 给出更紧的 Exp 下界，避免假阴性
      - 适用于零和、General-sum、非 LQ 等任意博弈
    """
    cfg     = trainer.cfg
    V_star  = trainer.env.analytic_value(cfg.x0)
    metrics = trainer.evaluate()
    J_cur   = metrics["J_mean"]

    if verbose:
        print("=" * 60)
        print("Exploitability 计算（Warm-start BR）")
        print("=" * 60)
        print(f"  当前策略 E[J] = {J_cur:.4f}")
        print(f"  V*            = {V_star:.4f}  "
              f"（仅供参考，Exp 不依赖此值）")
        print(f"  BR 初始化：Warm-start（复制当前策略权重）")

    # ── P1 BR（固定 α2*，用 α1* 初始化 BR）──────────────────
    if verbose:
        print(f"\n  [1/2] P1 best response（固定 α2*）...")
    _, J_br1, hist1, conv1 = compute_best_response_warmstart(
        fixed_actor   = trainer.actor2,
        br_actor_init = trainer.actor1,   # ← warm-start
        fixed_player  = 2,
        env=trainer.env, cfg=cfg, verbose=verbose)
    delta1 = max(J_cur - J_br1, 0.0)

    if verbose:
        print(f"  J(br1, α2*) = {J_br1:.4f}  "
              f"{'收敛 ✅' if conv1 else '未收敛 ⚠️'}")
        print(f"  δ1          = {delta1:.4f}  "
              f"（P1 通过偏离能减少的 J1）")

    # ── P2 BR（固定 α1*，用 α2* 初始化 BR）──────────────────
    if verbose:
        print(f"\n  [2/2] P2 best response（固定 α1*）...")
    _, J_br2, hist2, conv2 = compute_best_response_warmstart(
        fixed_actor   = trainer.actor1,
        br_actor_init = trainer.actor2,   # ← warm-start
        fixed_player  = 1,
        env=trainer.env, cfg=cfg, verbose=verbose)
    delta2 = max(J_br2 - J_cur, 0.0)

    if verbose:
        print(f"  J(α1*, br2) = {J_br2:.4f}  "
              f"{'收敛 ✅' if conv2 else '未收敛 ⚠️'}")
        print(f"  δ2          = {delta2:.4f}  "
              f"（P2 通过偏离能增加的 J2）")

    exploitability = (delta1 + delta2) / 2.0
    result = {
        "J_cur":          J_cur,
        "J_br1":          J_br1,
        "J_br2":          J_br2,
        "delta1":         delta1,
        "delta2":         delta2,
        "exploitability": exploitability,
        "br1_converged":  conv1,
        "br2_converged":  conv2,
        "br1_J_history":  hist1,
        "br2_J_history":  hist2,
        "V_star":         V_star,
        "val_error":      abs(J_cur - V_star),
    }

    if verbose:
        print(f"\n{'='*60}")
        if not conv1 or not conv2:
            print(f"  ⚠️  BR 未完全收敛，Exp 是松下界")
            print(f"      建议增大 br_episodes "
                  f"{cfg.br_episodes} → {cfg.br_episodes * 2}")
        print(f"  Exploitability = (δ1+δ2)/2 = {exploitability:.4f}")
        print(f"  {'✅ 收敛到 NE' if exploitability < 0.05 else '⚠️  未收敛到 NE'}")
        print(f"  ValErr = {result['val_error']:.4f}  "
              f"（参考值，不作为收敛判据）")
        if conv1 and conv2:
            print(f"  两个 BR 均已收敛，Exp 是可靠下界 ✅")
        print(f"{'='*60}")

    return result


# ── 在当前 trainer 上运行 Warm-start Exp ─────────────────────────
result_ws = compute_exploitability_warmstart(trainer, verbose=True)

## 9. Training Run

Config: Q=0 (pure terminal cost), N=5, x₀=2.  
Expected convergence: ValErr < 0.01, σ[u₂] ≈ 1.0 (Player 2 learns two-point mixed strategy).  
Note: Player 1's equilibrium is non-unique at Q=0 (Proposition 2.8 in the technical report) — any u₁(0) ∈ [−3,3] is valid. The algorithm converges to u₁(0) ≈ −1, which is a valid NE.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 主入口
# ══════════════════════════════════════════════════════════════════════
 
if __name__ == "__main__":
 
    # ── 验证 analytic_value 正确性 ───────────────────────────────
    _tests = [(0.0, 1.0), (0.1, 1.8), (1.0, 9.0)]
    for q, expected in _tests:
        got = LQGameEnv(Q=q, QN=1.0, N=5, c1=3.0, c2=1.0).analytic_value(2.0)
        assert abs(got - expected) < 1e-5, f"Q={q}: {got} ≠ {expected}"
    print(f"✅ analytic_value 验证通过: "
          f"Q=0→{_tests[0][1]}, Q=0.1→{_tests[1][1]}, Q=1→{_tests[2][1]}")
 
    # ── 验证方法挂载 ──────────────────────────────────────────────
    cfg_test = Config()
    t = Trainer(cfg_test)
    for _ in range(20):
        t._collect_trajectory(use_random=True)
    loss = t._update_critic(player=1)
    assert loss != 0.0, "❌ _update_critic 未正确挂载"
    m = t.evaluate()
    assert m["J_mean"] != 0.0, "❌ evaluate 未正确挂载"
    print(f"✅ 方法挂载验证通过  Critic loss={loss:.4f}")
    del t, cfg_test
 
    # ── 正式训练 ──────────────────────────────────────────────────
    cfg                 = Config()
    cfg.entropy_coef_p1 = 0.0    # P1 纯策略均衡，不加熵正则
    cfg.entropy_coef_p2 = 0.3    # P2 混合策略均衡，保持熵正则
    cfg.total_episodes  = 8000   # 延长训练保证 P2 充分收敛
 
    print(f"\n训练配置：")
    print(f"  Q={cfg.Q}  V*={cfg.Q*cfg.x0**2 + cfg.c2**2*((cfg.N-1)*cfg.Q+cfg.QN):.4f}")
    print(f"  entropy_coef_p1={cfg.entropy_coef_p1}  "
          f"entropy_coef_p2={cfg.entropy_coef_p2}")
    print(f"  total_episodes={cfg.total_episodes}")
 
    trainer = Trainer(cfg)
    trainer.warmup()
    trainer.train()
 
    # ── 最终评估 ──────────────────────────────────────────────────
    V_star  = trainer.env.analytic_value(cfg.x0)
    metrics = trainer.evaluate()
    print(f"\n{'='*60}")
    print(f"最终评估")
    print(f"{'='*60}")
    print(f"  V*     = {V_star:.4f}")
    print(f"  E[J]   = {metrics['J_mean']:.4f}")
    print(f"  ValErr = {metrics['value_error']:.4f}  "
          f"{'✅' if metrics['value_error'] < 0.1 else '⚠️'}")
    print(f"  P1 mean= {metrics['p1_u0_mean']:+.4f}  "
          f"(解析=-2.0)  "
          f"{'✅' if abs(metrics['p1_u0_mean']+2.0) < 0.3 else '⚠️'}")
    print(f"  P1 std = {metrics['p1_u0_std']:.4f}  "
          f"{'✅ 纯策略' if metrics['p1_u0_std'] < 0.1 else '⚠️ 仍有噪声'}")
    print(f"  P2 mean= {metrics['p2_u0_mean']:+.4f}  "
          f"{'✅' if abs(metrics['p2_u0_mean']) < 0.15 else '⚠️'}")
    print(f"  P2 std = {metrics['p2_u0_std']:.4f}  "
          f"{'✅' if abs(metrics['p2_u0_std']-1.0) < 0.15 else '⚠️'}")
 
    # ── Exploitability ────────────────────────────────────────────
    # 期望：P1 std≈0 → P2 BR 能找到更高 J → δ2>0 且 Exp<0.1
    result = compute_exploitability(trainer, verbose=True)
 
    print(f"\n{'='*60}")
    print(f"判断标准")
    print(f"{'='*60}")
    print(f"  P1 std={metrics['p1_u0_std']:.4f}  "
          f"{'✅ 纯策略' if metrics['p1_u0_std']<0.1 else '⚠️ 有噪声'}")
    print(f"  δ2 方向: J_br2={result['J_br2']:.4f} vs "
          f"J_cur={result['J_cur']:.4f}  "
          f"{'✅ 方向正确' if result['J_br2'] >= result['J_cur'] else '⚠️ 方向异常'}")
    print(f"  Exp={result['exploitability']:.4f}  "
          f"{'✅ 收敛到 NE' if result['exploitability']<0.1 else '⚠️ 未收敛'}")

## 10. Verification & Diagnostics

Confirms convergence is genuine NE and not an artefact of BR training failure.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Step 8: Exploitability（含 BR 收敛检测 + 双 GPU 设备修复）
# ══════════════════════════════════════════════════════════════════════
 
def compute_best_response(fixed_actor, fixed_player, env,
                           cfg, verbose=False):
    """
    固定一方策略，训练另一方的 best response Actor。
    理论基础：固定对手后博弈退化为单智能体 MDP，
              BR 训练 = 在该 MDP 上跑 AC，合法且理论收敛。
    返回：(br_actor, br_J, J_history, converged)
    """
    dev       = cfg.device
    br_player = 3 - fixed_player
    # 双 GPU 时 fixed_actor 可能在 cuda:1，自动检测其设备
    fixed_dev = next(fixed_actor.parameters()).device
 
    br_actor = Actor(
        noise_dim=cfg.noise_dim,
        action_bound=cfg.c1 if br_player == 1 else cfg.c2,
        time_dim=cfg.time_dim,
        hidden=cfg.hidden).to(dev)
    br_critic = DoubleQCritic(
        time_dim=cfg.time_dim, hidden=cfg.hidden).to(dev)
    br_critic_target = copy.deepcopy(br_critic)
    for p in br_critic_target.parameters():
        p.requires_grad_(False)
 
    opt_br_a = Adam(br_actor.parameters(),  lr=cfg.br_actor_lr)
    opt_br_c = Adam(br_critic.parameters(), lr=cfg.br_critic_lr)
    br_buf   = ReplayBuffer(
        capacity=cfg.buffer_capacity // 5,
        recent_ratio=cfg.recent_ratio, device=dev)
 
    def soft_update_br():
        for p_o, p_t in zip(br_critic.parameters(),
                             br_critic_target.parameters()):
            p_t.data.mul_(1.0 - cfg.tau)
            p_t.data.add_(cfg.tau * p_o.data)
 
    def collect(use_random=False):
        B = cfg.warmup_batch if use_random else cfg.train_batch
        x = env.reset(cfg.x0, B, dev)
        for k in range(env.N):
            if br_player == 1:
                u_br  = ((torch.rand(B, 1, device=dev)*2-1)*cfg.c1
                          if use_random else
                          br_actor.sample(x, k, env.N, dev))
                u_fix = fixed_actor.sample(
                    x.to(fixed_dev), k, env.N, fixed_dev).to(dev)
                u1, u2 = u_br, u_fix
            else:
                u_fix = fixed_actor.sample(
                    x.to(fixed_dev), k, env.N, fixed_dev).to(dev)
                u_br  = ((torch.rand(B, 1, device=dev)*2-1)*cfg.c2
                          if use_random else
                          br_actor.sample(x, k, env.N, dev))
                u1, u2 = u_fix, u_br
            x_next, r, done = env.step(x, u1, u2, k)
            br_buf.push(k, x, u1, u2, r, x_next, done)
            x = x_next
 
    def update_br_critic():
        if len(br_buf) < cfg.critic_batch:
            return 0.0
        batch    = br_buf.sample(cfg.critic_batch)
        k_float  = batch["k"]
        x_b      = batch["x"]
        r_b      = batch["r"]
        x_next_b = batch["x_next"]
        done_b   = batch["done"]
        B        = x_b.shape[0]
        u_self_b = batch["u1"] if br_player == 1 else batch["u2"]
        r_self   = r_b if br_player == 1 else -r_b
        k_int    = k_float.long()
        k_next   = (k_float + 1).long().clamp(max=env.N - 1)
        not_done = (done_b == 0.0)
        emb_cache  = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                      for kv in range(env.N)}
        k_emb_cur  = torch.stack(
            [emb_cache[kv.item()] for kv in k_int])
        k_emb_next = torch.stack(
            [emb_cache[kv.item()] for kv in k_next])
        with torch.no_grad():
            g_next = torch.zeros(B, 1, device=dev)
            s_next = torch.zeros(B, 1, device=dev)
            u_next = torch.zeros(B, 1, device=dev)
            if not_done.any():
                for k_val in k_next[not_done].unique():
                    mask  = not_done & (k_next == k_val)
                    x_sub = x_next_b[mask]
                    g, s  = estimate_opponent_moments(
                        fixed_actor, x_sub.to(fixed_dev),
                        k=k_val.item(), N=env.N,
                        device=fixed_dev, K=cfg.K_moment)
                    g_next[mask] = g.to(dev)
                    s_next[mask] = s.to(dev)
                    u_next[mask] = br_actor.sample(
                        x_sub, k_val.item(), env.N, dev)
            q_next = br_critic_target.mean_Q(
                x_next_b, u_next, g_next, s_next, k_emb_next)
            target = r_self + (1.0 - done_b) * q_next
        g_cur = torch.zeros(B, 1, device=dev)
        s_cur = torch.zeros(B, 1, device=dev)
        with torch.no_grad():
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                g, s  = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    k=k_val.item(), N=env.N,
                    device=fixed_dev, K=cfg.K_moment)
                g_cur[mask] = g.to(dev)
                s_cur[mask] = s.to(dev)
        q1_p, q2_p = br_critic(
            x_b, u_self_b, g_cur, s_cur, k_emb_cur)
        loss = (F.huber_loss(q1_p, target) +
                F.huber_loss(q2_p, target))
        opt_br_c.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(br_critic.parameters(), 5.0)
        opt_br_c.step()
        return loss.item()
 
    def update_br_actor():
        if len(br_buf) < cfg.actor_batch:
            return 0.0
        batch  = br_buf.sample(cfg.actor_batch)
        k_int  = batch["k"].long()
        x_b    = batch["x"]
        B      = x_b.shape[0]
        emb_cache = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                     for kv in range(env.N)}
        k_emb_cur = torch.stack(
            [emb_cache[kv.item()] for kv in k_int])
        with torch.no_grad():
            g_opp = torch.zeros(B, 1, device=dev)
            s_opp = torch.zeros(B, 1, device=dev)
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                g, s  = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    k=k_val.item(), N=env.N,
                    device=fixed_dev, K=cfg.K_moment)
                g_opp[mask] = g.to(dev)
                s_opp[mask] = s.to(dev)
        for p in br_critic.parameters():
            p.requires_grad_(False)
        try:
            u_parts, idx_parts = [], []
            for k_val in k_int.unique():
                mask  = (k_int == k_val)
                idx   = mask.nonzero(as_tuple=True)[0]
                z_sub = torch.randn(
                    mask.sum(), br_actor.noise_dim, device=dev)
                u_parts.append(br_actor(
                    x_b[mask], z_sub, emb_cache[k_val.item()]))
                idx_parts.append(idx)
            all_idx = torch.cat(idx_parts)
            all_u   = torch.cat(u_parts)
            inv_idx = torch.empty_like(all_idx)
            inv_idx[all_idx] = torch.arange(B, device=dev)
            u_self  = all_u[inv_idx]
            q    = br_critic.conservative_Q(
                x_b, u_self, g_opp, s_opp,
                k_emb_cur, beta=cfg.beta_ucb)
            loss = -q.mean()
            opt_br_a.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(br_actor.parameters(), 2.0)
            opt_br_a.step()
        finally:
            for p in br_critic.parameters():
                p.requires_grad_(True)
        return loss.item()
 
    def eval_br_J(n=500):
        x_eval  = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                if br_player == 1:
                    u1 = br_actor.sample(x_eval, k, env.N, dev)
                    u2 = fixed_actor.sample(
                        x_eval.to(fixed_dev), k, env.N,
                        fixed_dev).to(dev)
                else:
                    u1 = fixed_actor.sample(
                        x_eval.to(fixed_dev), k, env.N,
                        fixed_dev).to(dev)
                    u2 = br_actor.sample(x_eval, k, env.N, dev)
                x_eval, r, _ = env.step(x_eval, u1, u2, k)
                total_r += r
        return (-total_r).mean().item()
 
    # ── Warmup ──────────────────────────────────────────────────
    for _ in range(min(cfg.warmup_episodes // 2, 50)):
        collect(use_random=True)
    for _ in range(cfg.br_critic_steps):
        update_br_critic()
        soft_update_br()
 
    # ── 主循环（含收敛检测）─────────────────────────────────────
    J_history = []
    converged = False
    w         = max(cfg.br_converge_window // 20, 1)
 
    for ep in range(cfg.br_episodes):
        collect(use_random=False)
        for _ in range(cfg.br_critic_updates):
            update_br_critic()
        soft_update_br()
        update_br_actor()
 
        if (ep + 1) % 20 == 0:
            j_val = eval_br_J()
            J_history.append(j_val)
            if len(J_history) >= w:
                recent  = J_history[-w:]
                j_range = max(recent) - min(recent)
                if j_range < cfg.br_converge_tol:
                    converged = True
                    if verbose:
                        print(f"    BR 收敛于 ep {ep+1}  "
                              f"J={j_val:.4f}  range={j_range:.4f}")
                    break
 
        if verbose and (ep + 1) % 200 == 0 and not converged:
            j_val = J_history[-1] if J_history else float("nan")
            print(f"    BR ep {ep+1}/{cfg.br_episodes}  J={j_val:.4f}")
 
    if verbose and not converged:
        print(f"    ⚠️  BR 未收敛（{cfg.br_episodes} ep）"
              f"  建议增大 br_episodes")
 
    # ── 最终精确评估 ─────────────────────────────────────────────
    br_J = eval_br_J(n=cfg.br_eval_episodes)
    return br_actor, br_J, J_history, converged
 
 
def compute_exploitability(trainer, verbose=True) -> dict:
    """
    计算 Exploitability：
      Exp = (δ1 + δ2) / 2
      δ1 = max(J_cur - J(br1, α2*), 0)   P1 还能少多少
      δ2 = max(J(α1*, br2) - J_cur, 0)   P2 还能多多少
 
    Exp = 0  ⟺  NE（不依赖解析解，通用指标）
    注：计算值是真实 Exp 的下界（BR 由有限步近似）
    """
    cfg     = trainer.cfg
    V_star  = trainer.env.analytic_value(cfg.x0)
    metrics = trainer.evaluate()
    J_cur   = metrics["J_mean"]
 
    if verbose:
        print("=" * 60)
        print("Exploitability 计算")
        print("=" * 60)
        print(f"  当前策略 E[J] = {J_cur:.4f}")
        print(f"  V*            = {V_star:.4f}")
        print(f"  注：Exp 是真实值的下界（BR 由有限步近似）")
 
    if verbose:
        print(f"\n  [1/2] P1 best response（固定 α2*）...")
    _, J_br1, hist1, conv1 = compute_best_response(
        trainer.actor2, fixed_player=2,
        env=trainer.env, cfg=cfg, verbose=verbose)
    delta1 = max(J_cur - J_br1, 0.0)
 
    if verbose:
        print(f"  J(br1, α2*) = {J_br1:.4f}  "
              f"{'收敛 ✅' if conv1 else '未收敛 ⚠️'}")
        print(f"  δ1          = {delta1:.4f}")
 
    if verbose:
        print(f"\n  [2/2] P2 best response（固定 α1*）...")
    _, J_br2, hist2, conv2 = compute_best_response(
        trainer.actor1, fixed_player=1,
        env=trainer.env, cfg=cfg, verbose=verbose)
    delta2 = max(J_br2 - J_cur, 0.0)
 
    if verbose:
        print(f"  J(α1*, br2) = {J_br2:.4f}  "
              f"{'收敛 ✅' if conv2 else '未收敛 ⚠️'}")
        print(f"  δ2          = {delta2:.4f}")
 
    exploitability = (delta1 + delta2) / 2.0
    result = {
        "J_cur":          J_cur,
        "J_br1":          J_br1,
        "J_br2":          J_br2,
        "delta1":         delta1,
        "delta2":         delta2,
        "exploitability": exploitability,
        "br1_converged":  conv1,
        "br2_converged":  conv2,
        "br1_J_history":  hist1,
        "br2_J_history":  hist2,
        "V_star":         V_star,
        "val_error":      abs(J_cur - V_star),
    }
 
    if verbose:
        print(f"\n{'='*60}")
        if not conv1 or not conv2:
            print(f"  ⚠️  BR 未完全收敛，Exp 是松下界")
            print(f"      建议增大 br_episodes "
                  f"{cfg.br_episodes} → {cfg.br_episodes * 2}")
        print(f"  Exploitability = (δ1+δ2)/2 = {exploitability:.4f}")
        print(f"  {'✅ 接近 NE' if exploitability < 0.1 else '⚠️  未收敛'}")
        print(f"  ValErr         = {result['val_error']:.4f}")
        if conv1 and conv2:
            consistent = abs(exploitability - result["val_error"]) < 0.2
            print(f"  Exp≈ValErr     = "
                  f"{'✅' if consistent else '⚠️  偏差大，检查 BR 质量'}")
        print(f"{'='*60}")
 
    return result
 

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 详细验证：确认 Exp=0 是真实 NE 而非 BR 训练失败
# ══════════════════════════════════════════════════════════════════════

def detailed_verification(trainer, n_mc=10000):
    cfg = trainer.cfg
    dev = trainer.dev1
    env = trainer.env
    V_star = env.analytic_value(cfg.x0)

    print("=" * 60)
    print("详细 NE 验证")
    print("=" * 60)
    print(f"  V* = {V_star:.4f}  x0={cfg.x0}")

    # ── 当前策略基准 J ─────────────────────────────────────────
    def eval_J(actor1, actor2, n=n_mc, label=""):
        x       = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                u1 = actor1.sample(x, k, env.N, dev)
                u2 = actor2.sample(
                    x.to(trainer.dev2), k, env.N,
                    trainer.dev2).to(dev) if actor2 is trainer.actor2 \
                    else actor2.sample(x, k, env.N, dev)
                x, r, _ = env.step(x, u1, u2, k)
                total_r += r
        j = (-total_r).mean().item()
        std = (-total_r).std().item()
        if label:
            print(f"  {label}: J={j:.4f} ± {std/n**0.5:.4f}")
        return j

    J_cur = eval_J(trainer.actor1, trainer.actor2,
                   label="当前策略 J(α1*, α2*)")

    # ══ 验证1：解析策略基准 ════════════════════════════════════
    print(f"\n[验证1] 解析策略基准")
    print(f"  理论：P1打u1=-x，P2打±c2两点分布，J应≈V*={V_star:.4f}")

    class AnalyticActor1:
        """P1 解析策略：u1 = -x（纯策略）"""
        def sample(self, x, k, N, device):
            return env.analytic_u1(x)

    class AnalyticActor2:
        """P2 解析策略：±c2 两点分布"""
        def sample(self, x, k, N, device):
            return env.analytic_u2_sample(x)

    J_analytic = eval_J(AnalyticActor1(), AnalyticActor2(),
                        label="解析策略 J(u1*=-x, u2*=±c2)")
    print(f"  误差 |J_analytic - V*| = {abs(J_analytic - V_star):.4f}  "
          f"{'✅' if abs(J_analytic - V_star) < 0.05 else '⚠️'}")

    # ══ 验证2：P1 策略检验（固定 α2*，扫描 P1 策略）══════════
    print(f"\n[验证2] P1 策略扫描（固定 α2*，看当前 α1* 是否最优）")
    print(f"  理论：P1最优策略是 u1=-x，任何偏离都应使 J 增大")

    class FixedP1:
        """固定动作的 P1 策略"""
        def __init__(self, u_val):
            self.u_val = u_val
        def sample(self, x, k, N, device):
            return torch.full_like(x, self.u_val)

    class ScaleP1:
        """u1 = scale * (-x) 的 P1 策略"""
        def __init__(self, scale):
            self.scale = scale
        def sample(self, x, k, N, device):
            return (-x * self.scale).clamp(-cfg.c1, cfg.c1)

    print(f"  {'策略':25s}  {'J':>8s}  {'vs α1*':>10s}")
    print(f"  {'-'*50}")

    # 当前 α1* 的 J
    j_p1_cur = eval_J(trainer.actor1, trainer.actor2)
    print(f"  {'当前 α1*':25s}  {j_p1_cur:>8.4f}  {'基准':>10s}")

    # 解析最优
    j_analytic_p1 = eval_J(AnalyticActor1(), trainer.actor2)
    better = "✅ α1* 更好" if j_p1_cur <= j_analytic_p1 else "⚠️ 解析更好"
    print(f"  {'u1=-x（解析最优）':25s}  {j_analytic_p1:>8.4f}  "
          f"{'Δ='+f'{j_analytic_p1-j_p1_cur:+.4f}':>10s} {better}")

    # 缩放策略
    for scale in [0.5, 0.8, 1.0, 1.2, 1.5]:
        j_s = eval_J(ScaleP1(scale), trainer.actor2)
        flag = "← 比α1*好！" if j_s < j_p1_cur - 0.01 else ""
        print(f"  {f'u1={scale:.1f}×(-x)':25s}  {j_s:>8.4f}  "
              f"{'Δ='+f'{j_s-j_p1_cur:+.4f}':>10s} {flag}")

    # 固定常数策略
    for u_val in [-3.0, -2.0, -1.0, 0.0, 1.0]:
        j_s = eval_J(FixedP1(u_val), trainer.actor2)
        flag = "← 比α1*好！" if j_s < j_p1_cur - 0.01 else ""
        print(f"  {f'u1≡{u_val:.1f}（固定）':25s}  {j_s:>8.4f}  "
              f"{'Δ='+f'{j_s-j_p1_cur:+.4f}':>10s} {flag}")

    # 把验证3这一段替换成下面的版本

    # ══ 验证3：P2 策略检验 ════════════════════════════════════
    print(f"\n[验证3] P2 策略扫描（固定 α1*，看当前 α2* 是否最优）")
    print(f"  理论：P2最优策略是±c2两点分布，任何偏离都应使 J 减小")

    class FixedP2:
        def __init__(self, u_val):
            self.u_val = u_val
        def sample(self, x, k, N, device):
            return torch.full_like(x, self.u_val)

    class BiasedP2:
        def __init__(self, p_pos):
            self.p_pos = p_pos
        def sample(self, x, k, N, device):
            mask = torch.rand_like(x) < self.p_pos
            return torch.where(mask,
                               torch.full_like(x,  cfg.c2),
                               torch.full_like(x, -cfg.c2))

    # ── 先算基准，参数传入避免闭包问题 ────────────────────────
    def eval_J_fix_p1(actor2_cand, baseline, n=n_mc, label=""):
        x       = env.reset(cfg.x0, n, dev)
        total_r = torch.zeros(n, device=dev)
        with torch.no_grad():
            for k in range(env.N):
                u1 = trainer.actor1.sample(x, k, env.N, dev)
                u2 = actor2_cand.sample(x, k, env.N, dev)
                x, r, _ = env.step(x, u1, u2, k)
                total_r += r
        j = (-total_r).mean().item()
        if label:
            flag = "← 比α2*好！" if j > baseline + 0.01 else ""
            print(f"  {label:30s}  {j:>8.4f}  "
                  f"{'Δ='+f'{j-baseline:+.4f}':>10s} {flag}")
        return j

    class CurrentA2:
        def sample(self, x, k, N, device):
            return trainer.actor2.sample(
                x.to(trainer.dev2), k, N,
                trainer.dev2).to(dev)

    # 先算基准
    j_cur_p2 = eval_J_fix_p1(CurrentA2(), baseline=0,
                               label=None)
    print(f"  {'当前 α2*':30s}  {j_cur_p2:>8.4f}  {'基准':>10s}")
    print(f"  {'-'*55}")

    # 再扫描各策略，传入基准值
    eval_J_fix_p1(AnalyticActor2(), j_cur_p2,
                  label="u2=±c2（解析最优）")
    for u_val in [-1.0, -0.5, 0.0, 0.5, 1.0]:
        eval_J_fix_p1(FixedP2(u_val), j_cur_p2,
                      label=f"u2≡{u_val:.1f}（固定）")
    for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
        eval_J_fix_p1(BiasedP2(p), j_cur_p2,
                      label=f"u2=±c2, P(+c2)={p:.1f}")

    # ══ 验证4：Warm-start BR（解决 BR 训练失败问题）════════════
    print(f"\n[验证4] Warm-start BR（用 α2* 初始化，继续寻找更优策略）")
    print(f"  目的：排除 BR 从随机初始化导致局部最优的可能")

    # 用 α2* 的权重初始化 BR actor
    br_actor2_ws = Actor(
        noise_dim=cfg.noise_dim,
        action_bound=cfg.c2,
        time_dim=cfg.time_dim,
        hidden=cfg.hidden).to(dev)
    # 把 actor2 的权重复制过来（注意设备迁移）
    state = {k: v.to(dev) for k, v in
             trainer.actor2.state_dict().items()}
    br_actor2_ws.load_state_dict(state)

    br_critic_ws = DoubleQCritic(
        time_dim=cfg.time_dim, hidden=cfg.hidden).to(dev)
    br_critic_ws_target = copy.deepcopy(br_critic_ws)
    for p in br_critic_ws_target.parameters():
        p.requires_grad_(False)

    opt_ws_a = Adam(br_actor2_ws.parameters(), lr=cfg.br_actor_lr)
    opt_ws_c = Adam(br_critic_ws.parameters(), lr=cfg.br_critic_lr)

    fixed_actor = trainer.actor1
    fixed_dev   = trainer.dev1
    br_buf_ws   = ReplayBuffer(
        capacity=cfg.buffer_capacity // 5,
        recent_ratio=cfg.recent_ratio, device=dev)

    # 收集数据
    for _ in range(50):
        x = env.reset(cfg.x0, cfg.train_batch, dev)
        for k in range(env.N):
            u1  = fixed_actor.sample(x, k, env.N, dev)
            u2  = br_actor2_ws.sample(x, k, env.N, dev)
            x_n, r, done = env.step(x, u1, u2, k)
            br_buf_ws.push(k, x, u1, u2, r, x_n, done)
            x = x_n

    def soft_update_ws():
        for p_o, p_t in zip(br_critic_ws.parameters(),
                             br_critic_ws_target.parameters()):
            p_t.data.mul_(1.0 - cfg.tau)
            p_t.data.add_(cfg.tau * p_o.data)

    # Warmup critic
    for _ in range(200):
        if len(br_buf_ws) < cfg.critic_batch:
            break
        batch    = br_buf_ws.sample(
            min(cfg.critic_batch, len(br_buf_ws)))
        k_float  = batch["k"]
        x_b      = batch["x"]
        r_b      = batch["r"]
        x_next_b = batch["x_next"]
        done_b   = batch["done"]
        u2_b     = batch["u2"]
        r_self   = -r_b
        B        = x_b.shape[0]
        k_int    = k_float.long()
        k_next   = (k_float+1).long().clamp(max=env.N-1)
        not_done = (done_b == 0.0)
        emb_cache = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                     for kv in range(env.N)}
        k_emb_c = torch.stack([emb_cache[kv.item()] for kv in k_int])
        k_emb_n = torch.stack([emb_cache[kv.item()] for kv in k_next])
        with torch.no_grad():
            g1n = torch.zeros(B, 1, device=dev)
            s1n = torch.zeros(B, 1, device=dev)
            u2n = torch.zeros(B, 1, device=dev)
            if not_done.any():
                for kv in k_next[not_done].unique():
                    mask = not_done & (k_next == kv)
                    g, s = estimate_opponent_moments(
                        fixed_actor, x_next_b[mask].to(fixed_dev),
                        kv.item(), env.N, fixed_dev, cfg.K_moment)
                    g1n[mask] = g.to(dev)
                    s1n[mask] = s.to(dev)
                    u2n[mask] = br_actor2_ws.sample(
                        x_next_b[mask], kv.item(), env.N, dev)
            q_n = br_critic_ws_target.mean_Q(
                x_next_b, u2n, g1n, s1n, k_emb_n)
            tgt = r_self + (1.0 - done_b) * q_n
        g1c = torch.zeros(B, 1, device=dev)
        s1c = torch.zeros(B, 1, device=dev)
        with torch.no_grad():
            for kv in k_int.unique():
                mask = (k_int == kv)
                g, s = estimate_opponent_moments(
                    fixed_actor, x_b[mask].to(fixed_dev),
                    kv.item(), env.N, fixed_dev, cfg.K_moment)
                g1c[mask] = g.to(dev)
                s1c[mask] = s.to(dev)
        q1, q2 = br_critic_ws(x_b, u2_b, g1c, s1c, k_emb_c)
        loss = F.huber_loss(q1, tgt) + F.huber_loss(q2, tgt)
        opt_ws_c.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(br_critic_ws.parameters(), 5.0)
        opt_ws_c.step()
        soft_update_ws()

    # 继续训练 actor
    J_ws_history = []
    for ep in range(400):
        x = env.reset(cfg.x0, cfg.train_batch, dev)
        for k in range(env.N):
            u1  = fixed_actor.sample(x, k, env.N, dev)
            u2  = br_actor2_ws.sample(x, k, env.N, dev)
            x_n, r, done = env.step(x, u1, u2, k)
            br_buf_ws.push(k, x, u1, u2, r, x_n, done)
            x = x_n
        if len(br_buf_ws) >= cfg.critic_batch:
            for _ in range(10):
                # critic update（简化版）
                batch  = br_buf_ws.sample(cfg.critic_batch)
                k_int2 = batch["k"].long()
                x_b2   = batch["x"]
                u2_b2  = batch["u2"]
                r_self2= -batch["r"]
                B2     = x_b2.shape[0]
                emb_c2 = {kv: time_embed(kv, env.N, cfg.time_dim, dev)
                          for kv in range(env.N)}
                ke2    = torch.stack(
                    [emb_c2[kv.item()] for kv in k_int2])
                g1c2   = torch.zeros(B2, 1, device=dev)
                s1c2   = torch.zeros(B2, 1, device=dev)
                with torch.no_grad():
                    for kv in k_int2.unique():
                        mask = (k_int2 == kv)
                        g, s = estimate_opponent_moments(
                            fixed_actor, x_b2[mask].to(fixed_dev),
                            kv.item(), env.N, fixed_dev, cfg.K_moment)
                        g1c2[mask] = g.to(dev)
                        s1c2[mask] = s.to(dev)
                    q_t = br_critic_ws_target.mean_Q(
                        x_b2, u2_b2, g1c2, s1c2, ke2)
                    tgt2 = r_self2 + q_t
                q1_2, q2_2 = br_critic_ws(
                    x_b2, u2_b2, g1c2, s1c2, ke2)
                l2 = (F.huber_loss(q1_2, tgt2) +
                      F.huber_loss(q2_2, tgt2))
                opt_ws_c.zero_grad(); l2.backward()
                nn.utils.clip_grad_norm_(
                    br_critic_ws.parameters(), 5.0)
                opt_ws_c.step()
            soft_update_ws()
            # actor update
            for p in br_critic_ws.parameters():
                p.requires_grad_(False)
            try:
                batch_a = br_buf_ws.sample(cfg.actor_batch)
                ki_a    = batch_a["k"].long()
                xa      = batch_a["x"]
                Ba      = xa.shape[0]
                emb_a   = {kv: time_embed(
                    kv, env.N, cfg.time_dim, dev)
                           for kv in range(env.N)}
                ke_a    = torch.stack(
                    [emb_a[kv.item()] for kv in ki_a])
                go = torch.zeros(Ba, 1, device=dev)
                so = torch.zeros(Ba, 1, device=dev)
                with torch.no_grad():
                    for kv in ki_a.unique():
                        mask = (ki_a == kv)
                        g, s = estimate_opponent_moments(
                            fixed_actor, xa[mask].to(fixed_dev),
                            kv.item(), env.N, fixed_dev,
                            cfg.K_moment)
                        go[mask] = g.to(dev)
                        so[mask] = s.to(dev)
                u_p, i_p = [], []
                for kv in ki_a.unique():
                    mask = (ki_a == kv)
                    idx  = mask.nonzero(as_tuple=True)[0]
                    z    = torch.randn(
                        mask.sum(), br_actor2_ws.noise_dim,
                        device=dev)
                    u_p.append(br_actor2_ws(
                        xa[mask], z, emb_a[kv.item()]))
                    i_p.append(idx)
                ai  = torch.cat(i_p)
                au  = torch.cat(u_p)
                inv = torch.empty_like(ai)
                inv[ai] = torch.arange(Ba, device=dev)
                us  = au[inv]
                q_a = br_critic_ws.conservative_Q(
                    xa, us, go, so, ke_a, beta=cfg.beta_ucb)
                la  = -q_a.mean()
                opt_ws_a.zero_grad(); la.backward()
                nn.utils.clip_grad_norm_(
                    br_actor2_ws.parameters(), 2.0)
                opt_ws_a.step()
            finally:
                for p in br_critic_ws.parameters():
                    p.requires_grad_(True)

        if (ep + 1) % 100 == 0:
            x_e    = env.reset(cfg.x0, 1000, dev)
            tot_r  = torch.zeros(1000, device=dev)
            with torch.no_grad():
                for k in range(env.N):
                    u1_e = fixed_actor.sample(x_e, k, env.N, dev)
                    u2_e = br_actor2_ws.sample(x_e, k, env.N, dev)
                    x_e, r_e, _ = env.step(x_e, u1_e, u2_e, k)
                    tot_r += r_e
            j_ws = (-tot_r).mean().item()
            J_ws_history.append(j_ws)
            print(f"    Warm-start BR ep={ep+1}  J={j_ws:.4f}  "
                  f"{'> α2* ✅' if j_ws > J_cur + 0.01 else '≤ α2* (α2*已最优)'}")

    j_ws_final = J_ws_history[-1] if J_ws_history else float('nan')
    ws_delta2  = max(j_ws_final - J_cur, 0.0)
    print(f"\n  Warm-start BR 最终 J = {j_ws_final:.4f}")
    print(f"  Warm-start δ2        = {ws_delta2:.4f}  "
          f"{'✅ α2* 已是真实最优' if ws_delta2 < 0.05 else '⚠️ α2* 还有提升空间'}")

    # ══ 验证5：各时间步状态和动作分析 ═════════════════════════
    print(f"\n[验证5] 各时间步均值分析")
    print(f"  {'k':>3} | {'E[x(k)]':>10} | "
          f"{'E[u1(k)]':>10} | {'E[u2(k)]':>10} | "
          f"{'E[x(k+1)]':>10}")
    print(f"  {'-'*55}")
    x_t    = env.reset(cfg.x0, n_mc, dev)
    with torch.no_grad():
        for k in range(env.N):
            ex  = x_t.mean().item()
            u1k = trainer.actor1.sample(x_t, k, env.N, dev)
            u2k = trainer.actor2.sample(
                x_t.to(trainer.dev2), k, env.N,
                trainer.dev2).to(dev)
            eu1 = u1k.mean().item()
            eu2 = u2k.mean().item()
            x_t, _, _ = env.step(x_t, u1k, u2k, k)
            ex_next = x_t.mean().item()
            print(f"  {k:>3} | {ex:>10.4f} | "
                  f"{eu1:>10.4f} | {eu2:>10.4f} | "
                  f"{ex_next:>10.4f}")
    print(f"  理论：E[x(1)]=E[u2(0)]≈0，之后保持≈0")

    # ══ 综合判断 ═══════════════════════════════════════════════
    print(f"\n{'='*60}")
    print(f"综合判断")
    print(f"{'='*60}")
    checks = {
        "ValErr < 0.05":
            abs(J_cur - V_star) < 0.05,
        "P1 std < 0.05（纯策略）":
            trainer.evaluate()['p1_u0_std'] < 0.05,
        "P2 std > 0.95（混合策略）":
            trainer.evaluate()['p2_u0_std'] > 0.95,
        "解析 P1 不优于当前 α1*":
            j_analytic_p1 >= j_p1_cur - 0.01,
        "Warm-start δ2 < 0.05":
            ws_delta2 < 0.05,
    }
    all_pass = True
    for name, passed in checks.items():
        print(f"  {'✅' if passed else '❌'} {name}")
        if not passed:
            all_pass = False
    print(f"\n  {'🎉 所有验证通过，Exp=0 是真实 NE' if all_pass else '⚠️ 存在未通过项，需要进一步分析'}")
    return checks


# ── 执行 ──────────────────────────────────────────────────────────
checks = detailed_verification(trainer)
